# ACTRIS Aerosol Remote Sensing (automatic version)
<p style="margin-top: 10px;">
  <img src="https://tools.actris-ares.eu/wp-content/uploads/2025/07/Actris-150x150.png";">
</p>

This notebook provides tools for exploring aerosol remote sensing data of ACTRIS/EARLINET.
It downloads and plots aerosol optical product profiles, and downloads and visualizes corresponding
aerosol typing products.

**Automatic version**: no filenames or measurement IDs are hardcoded.
Everything is discovered from the metadata returned by the API, and the choices you have to make
(station, measurement, products to plot) are done through dropdowns with a confirm button rather than
by editing code. You only need to set the date/time range in the "Configuration" cell.

**Folder management: to avoid accumulating files across repeated runs, this notebook empties its `data/` and `exports/` folders at the start of each run. Use the export cell at the end to keep anything you want before running it again.**

> **Clearing the outputs.** Use *Edit → Clear Outputs of All Cells* in two situations:
>
> * **before running the notebook again**, so that the interactive widgets are rebuilt from scratch.
>   Widget state belongs to the kernel that created it, so leftovers from a previous run can make
>   controls stop responding or disappear;
> * **before saving**, so that the file stays small and does not carry stale widget state. Plotly
>   embeds a large JavaScript bundle into saved outputs, and reopening a notebook that stores widget
>   state shows "Error displaying widget: model not found".
>
> If controls are still unresponsive after clearing, restart the kernel (*Kernel → Restart Kernel*)
> and, should that not be enough, reload the browser page: the widgets live partly in the page
> itself, and a reload is sometimes the only way to clear what is left there.

### Import libraries

### Dependencies

This notebook needs a few third-party packages. In the D4Science VRE the shared workspace shares
files, not the Python environment: each user has their own container, so a notebook that runs for
one person can fail for another with `ModuleNotFoundError`.

The cell below installs only the packages that are actually missing, so it costs nothing on an
environment that already has them.

In [ ]:
# The kernel keeps a handle on the directory it was started in. If that directory is replaced
# while the kernel is running - the shared workspace being republished, for instance - every
# relative path fails with "Stale file handle" or FileNotFoundError. Recover instead of leaving
# the user with an error that has nothing to do with what they were doing.
import os

try:
    os.getcwd()
except OSError:
    _nb = os.environ.get("JPY_SESSION_NAME", "")
    _target = os.path.dirname(_nb) if _nb else os.path.expanduser("~")
    if not os.path.isdir(_target):
        _target = os.path.expanduser("~")
    os.chdir(_target)
    print(f"The working directory had become stale and was reset to {_target}.")
    print("If anything still looks wrong, restart the kernel (Kernel -> Restart Kernel).")

import os
import importlib.util
import subprocess
import sys

# module name -> pip package name
REQUIRED = {
    "numpy": "numpy",
    "requests": "requests",
    "xarray": "xarray",
    "netCDF4": "netCDF4",        # backend xarray uses to read the NetCDF products
    "matplotlib": "matplotlib",
    "plotly": "plotly",
    "ipywidgets": "ipywidgets",
    # Hidden dependency: plotly needs nbformat to render figures through the mime-type
    # renderer, but does not declare it as a hard requirement. A full Jupyter environment
    # always has it, so the omission only shows up in a clean one.
    "nbformat": "nbformat",
}

missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]

if not missing:
    print("--> All required packages are already installed.")
else:
    print(f"--> Installing missing packages: {', '.join(missing)}")
    # sys.executable, not '!pip': the shell's pip may belong to a different Python than the
    # one running this kernel, which is a classic source of "I installed it but it isn't found".
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                            capture_output=True, text=True,
                            cwd=os.path.expanduser("~"))
    if result.returncode == 0:
        print("--> Done. If an import still fails below, restart the kernel and run again.")
    else:
        print("--> Installation failed:")
        print(result.stderr[-1500:])
        if "externally-managed-environment" in result.stderr:
            print("\n--> This environment does not allow pip to install into it directly.")
            print("    Ask the VRE administrators to add the packages, or retry with:")
            print(f"    {sys.executable} -m pip install --user " + " ".join(missing))

In [ ]:
# Identity of this notebook. The file itself carries no version number - only one version is
# published at a time - so the version is printed here instead, which is also what tells you
# whether the kernel is running the file you think it is.
NOTEBOOK_NAME = "ACTRIS Aerosol Remote Sensing"
NOTEBOOK_VERSION = "4.3"
NOTEBOOK_DATE = "2026-11-22"

# What the notebook will produce; the menu in "What to analyse" sets it. The default is kept
# here so that a later cell re-run on its own still finds the flag defined.
TYPING_ONLY = False

print(f"{NOTEBOOK_NAME} - version {NOTEBOOK_VERSION} ({NOTEBOOK_DATE})")

from pathlib import Path
import shutil
import re
import glob
import os
import zipfile
import datetime
import math

import numpy as np
import requests
import xarray as xr

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.ticker import MaxNLocator, MultipleLocator

from ipywidgets import (IntProgress, Dropdown, Layout, Button, Output, SelectMultiple,
                        DatePicker, Text, HBox, VBox, Checkbox)
from IPython.display import display, HTML

# Bump this whenever the notebook is updated, so you can tell at a glance which version the kernel
# is actually running (a stale kernel keeps old function definitions even after the file is updated).


### Select station

Pick a **country** first, then a **station** from that country. Two extra groups sit at the top of the
first dropdown: **"All stations"** to browse the full list, and **"Mobile stations"** for the mobile
(non-fixed) platforms such as OCEANET and LACROS — these are kept out of the per-country lists, since
they aren't tied to a fixed site.

The map below shows the stations of the selected group, with the currently highlighted station in red —
it updates as you change the dropdowns. When you're happy with the choice, click **"Confirm selection"**
and check that the "Confirmed station: ..." message appears.

⚠️ Just picking a value in the dropdowns is not enough - you must click the confirm button, otherwise
the notebook falls back to the default example station (`POT` - Potenza, Italy), telling you so when it does.
Re-running this cell keeps an already-confirmed station.

In [ ]:
r = requests.get("https://api.actris-ares.eu/api/services/restapi/stations")
r.raise_for_status()
stations_list = r.json()

try:
    import plotly.graph_objects as go
    _HAS_PLOTLY = True
except ImportError:
    _HAS_PLOTLY = False

stations_by_id = {s.get("ID"): s for s in stations_list}

def _country_name(code):
    """Human-readable country name, taken from the 'City, Country' part of any Location field."""
    for s in stations_list:
        if s.get("countryCode") == code and not _is_mobile(s):
            loc = s.get("Location") or ""
            if "," in loc:
                return loc.rsplit(",", 1)[1].strip()
    return code

def _is_mobile(s):
    """Mobile (non-fixed) platforms are flagged by the word 'mobile' in their Location field."""
    return "mobile" in (s.get("Location") or "").lower()

def _group_label(code):
    if code == "ALL":
        return "all stations"
    if code == "MOBILE":
        return "mobile platforms"
    return _country_name(code)

def _stations_for(code):
    if code == "ALL":
        sel = list(stations_list)
    elif code == "MOBILE":
        sel = [s for s in stations_list if _is_mobile(s)]
    else:
        # Fixed stations only: mobile platforms live in their own "Mobile stations" group
        sel = [s for s in stations_list if s.get("countryCode") == code and not _is_mobile(s)]
    return sorted(
        (f"{s.get('ID')} - {s.get('Location')} - {s.get('Institute')}", s.get("ID")) for s in sel
    )

_fixed_countries = sorted({
    s.get("countryCode") for s in stations_list if s.get("countryCode") and not _is_mobile(s)
})
country_options = [("All stations", "ALL")]
if any(_is_mobile(s) for s in stations_list):
    country_options.append(("Mobile stations", "MOBILE"))
country_options += [(f"{c} - {_country_name(c)}", c) for c in _fixed_countries]

# Preserve an already-confirmed station if this cell gets re-run
if "SELECTED_STATION" not in globals():
    SELECTED_STATION = None
_initial_station = SELECTED_STATION if SELECTED_STATION in stations_by_id else "POT"
if _initial_station not in stations_by_id:
    _initial_station = stations_list[0].get("ID")
_initial_country = ("MOBILE" if _is_mobile(stations_by_id[_initial_station])
                    else stations_by_id[_initial_station].get("countryCode"))

country_dropdown = Dropdown(
    options=country_options, value=_initial_country, description="Country / group:",
    style={"description_width": "initial"}, layout=Layout(width="400px"),
)
station_dropdown = Dropdown(
    options=_stations_for(_initial_country), value=_initial_station, description="Station:",
    style={"description_width": "initial"}, layout=Layout(width="900px"),
)
confirm_button = Button(description="Confirm selection", button_style="success", icon="check")
confirm_output = Output()
map_output = Output()

def _draw_map(*_):
    """Show the stations of the selected country, highlighting the currently selected one."""
    with map_output:
        map_output.clear_output(wait=True)
        if not _HAS_PLOTLY:
            print("(plotly not available - install it to see the map)")
            return
        code = country_dropdown.value
        _ids = {v for _, v in _stations_for(code)}
        shown = [s for s in stations_list if s.get("ID") in _ids]
        current = station_dropdown.value
        others = [s for s in shown if s.get("ID") != current]
        sel = [s for s in shown if s.get("ID") == current]

        fig = go.Figure()
        if others:
            fig.add_trace(go.Scattergeo(
                lon=[s.get("Longitude") for s in others],
                lat=[s.get("Latitude") for s in others],
                text=[f"{s.get('ID')} - {s.get('Location')}" for s in others],
                hoverinfo="text", mode="markers", name="Other stations",
                marker=dict(size=8, color="#3B4872", line=dict(width=1, color="white")),
            ))
        if sel:
            fig.add_trace(go.Scattergeo(
                lon=[s.get("Longitude") for s in sel],
                lat=[s.get("Latitude") for s in sel],
                text=[f"{s.get('ID')} - {s.get('Location')}<br>{s.get('Institute')}" for s in sel],
                hoverinfo="text", mode="markers+text", name="Selected",
                textposition="top center", texttemplate="%{text}",
                marker=dict(size=15, color="crimson", line=dict(width=1.5, color="white")),
            ))
        # Scattergeo uses built-in geometry (no external map tiles needed)
        fig.update_geos(fitbounds="locations", resolution=50, showcountries=True,
                        countrycolor="#888", showland=True, landcolor="#f2efe9",
                        showocean=True, oceancolor="#dbe9f2", showlakes=False)
        fig.update_layout(height=420, margin=dict(l=0, r=0, t=30, b=0),
                          showlegend=False,
                          title=f"ACTRIS aerosol remote sensing stations - "
                                f"{_group_label(code)}")
        fig.show()

def _on_country_change(change):
    """Repopulate the station list when the country changes."""
    opts = _stations_for(change["new"])
    station_dropdown.unobserve(_on_station_change, names="value")
    station_dropdown.options = opts
    if opts:
        station_dropdown.value = opts[0][1]
    station_dropdown.observe(_on_station_change, names="value")
    _draw_map()

def _on_station_change(change):
    _draw_map()

def _on_confirm_clicked(b):
    global SELECTED_STATION
    SELECTED_STATION = station_dropdown.value
    with confirm_output:
        confirm_output.clear_output()
        s = stations_by_id.get(SELECTED_STATION, {})
        print(f"Confirmed station: {SELECTED_STATION} ({s.get('Location', '')})")

country_dropdown.observe(_on_country_change, names="value")
station_dropdown.observe(_on_station_change, names="value")
confirm_button.on_click(_on_confirm_clicked)

display(VBox([country_dropdown, station_dropdown, confirm_button, confirm_output, map_output]))
_draw_map()

if SELECTED_STATION:
    with confirm_output:
        print(f"Confirmed station: {SELECTED_STATION} (already confirmed - you can move to the next cell)")


### Select the date and time range

Pick the start/end date and time below and click **"Confirm date range"**, so you don't have to edit
any date string by hand in the code.

The defaults are set to the working example case used throughout this notebook — **Potenza (POT),
5 June 2025, 19:00:00 → 23:59:00** — so if you confirm nothing at all, the notebook still runs on a
known-good case. As with the other selectors, re-running this cell keeps an already-confirmed range.

In [ ]:
# TimePicker exists in ipywidgets >= 8; fall back to plain text boxes on older versions
try:
    from ipywidgets import TimePicker
    _HAS_TIMEPICKER = True
except ImportError:
    _HAS_TIMEPICKER = False

# Defaults = the working example case (Potenza, 5 June 2025, 19:00 -> 23:59)
DEFAULT_FROM_DATE = "2025-06-05"
DEFAULT_FROM_TIME = "19:00:00"
DEFAULT_TO_DATE = "2025-06-05"
DEFAULT_TO_TIME = "23:59:00"

if "SELECTED_DATE_RANGE" not in globals():
    SELECTED_DATE_RANGE = None
_init = SELECTED_DATE_RANGE or (DEFAULT_FROM_DATE, DEFAULT_FROM_TIME, DEFAULT_TO_DATE, DEFAULT_TO_TIME)

def _mk_time_widget(desc, value_str):
    if _HAS_TIMEPICKER:
        return TimePicker(
            description=desc,
            value=datetime.datetime.strptime(value_str, "%H:%M:%S").time(),
            style={"description_width": "initial"},
            layout=Layout(width="260px"),
        )
    return Text(
        description=desc,
        value=value_str,
        placeholder="HH:MM:SS",
        style={"description_width": "initial"},
        layout=Layout(width="260px"),
    )

def _read_time(widget):
    """Return the widget's value as an 'HH:MM:SS' string, whichever widget type it is."""
    v = widget.value
    if isinstance(v, str):
        return datetime.datetime.strptime(v.strip(), "%H:%M:%S").strftime("%H:%M:%S")
    return v.strftime("%H:%M:%S")

from_date_picker = DatePicker(
    description="From date:",
    value=datetime.datetime.strptime(_init[0], "%Y-%m-%d").date(),
    style={"description_width": "initial"}, layout=Layout(width="260px"),
)
from_time_picker = _mk_time_widget("From time:", _init[1])
to_date_picker = DatePicker(
    description="To date:",
    value=datetime.datetime.strptime(_init[2], "%Y-%m-%d").date(),
    style={"description_width": "initial"}, layout=Layout(width="260px"),
)
to_time_picker = _mk_time_widget("To time:", _init[3])

confirm_dates_button = Button(description="Confirm date range", button_style="success", icon="check")
confirm_dates_output = Output()

def _on_confirm_dates(b):
    global SELECTED_DATE_RANGE
    with confirm_dates_output:
        confirm_dates_output.clear_output()
        try:
            fd = from_date_picker.value.strftime("%Y-%m-%d")
            td = to_date_picker.value.strftime("%Y-%m-%d")
            ft = _read_time(from_time_picker)
            tt = _read_time(to_time_picker)
        except (AttributeError, ValueError):
            print("Invalid date/time: please make sure both dates are set and times are in HH:MM:SS format.")
            return
        if f"{td} {tt}" < f"{fd} {ft}":
            print("The end of the range is earlier than its start - please fix it and confirm again.")
            return
        SELECTED_DATE_RANGE = (fd, ft, td, tt)
        print(f"Confirmed range: {fd} {ft}  ->  {td} {tt}")

confirm_dates_button.on_click(_on_confirm_dates)

display(VBox([HBox([from_date_picker, from_time_picker]),
              HBox([to_date_picker, to_time_picker]),
              confirm_dates_button, confirm_dates_output]))

with confirm_dates_output:
    if SELECTED_DATE_RANGE:
        print(f"Confirmed range: {SELECTED_DATE_RANGE[0]} {SELECTED_DATE_RANGE[1]}  ->  "
              f"{SELECTED_DATE_RANGE[2]} {SELECTED_DATE_RANGE[3]} (already confirmed - you can move on)")
    else:
        print(f"Not confirmed yet - will default to {DEFAULT_FROM_DATE} {DEFAULT_FROM_TIME} -> "
              f"{DEFAULT_TO_DATE} {DEFAULT_TO_TIME} (example case).")


### Configuration

This cell just collects the choices confirmed in the selector cells above. If you skipped a selector
(or forgot to click its confirm button), it falls back to the working example case — **Potenza (POT),
5 June 2025, 19:00:00 → 23:59:00** — and prints a message saying so, so the notebook always runs on
something valid.

**Data folder management**: to avoid accumulating files across repeated runs, this notebook wipes the
whole `data/` folder at the start of each run (set `CLEAN_DATA_FOLDER = False` to disable this) and then
creates a fresh subfolder for this run only, named `data/{STATION}_{FROM}_{TO}/`. All the files this run
produces — downloaded NetCDF files, the MAC,HIRAC and HETEAC-Flex zip archives, extracted files, and the saved
plot — are kept together inside that subfolder.

In [ ]:
# Everything below falls back to the working example case (Potenza, 5 June 2025, 19:00 -> 23:59)
# if you didn't confirm a choice in the selector cells above.
DEFAULT_STATION = "POT"
DEFAULT_FROM_DATE = globals().get("DEFAULT_FROM_DATE", "2025-06-05")
DEFAULT_FROM_TIME = globals().get("DEFAULT_FROM_TIME", "19:00:00")
DEFAULT_TO_DATE = globals().get("DEFAULT_TO_DATE", "2025-06-05")
DEFAULT_TO_TIME = globals().get("DEFAULT_TO_TIME", "23:59:00")

SELECTED_STATION = globals().get("SELECTED_STATION")
SELECTED_DATE_RANGE = globals().get("SELECTED_DATE_RANGE")

if SELECTED_STATION:
    STATION = SELECTED_STATION
else:
    STATION = DEFAULT_STATION
    print(f"No station confirmed - defaulting to the example station: {STATION}")

if SELECTED_DATE_RANGE:
    FROM_DATE, FROM_TIME, TO_DATE, TO_TIME = SELECTED_DATE_RANGE
else:
    FROM_DATE, FROM_TIME = DEFAULT_FROM_DATE, DEFAULT_FROM_TIME
    TO_DATE, TO_TIME = DEFAULT_TO_DATE, DEFAULT_TO_TIME
    print(f"No date range confirmed - defaulting to the example range: "
          f"{FROM_DATE} {FROM_TIME} -> {TO_DATE} {TO_TIME}")

print(f"Search parameters: station={STATION}, {FROM_DATE} {FROM_TIME} -> {TO_DATE} {TO_TIME}")

# Wipe the whole "data" folder before this run, so it doesn't grow indefinitely across executions.
# Set to False if you'd rather keep results from previous runs.
CLEAN_DATA_FOLDER = True

DATA_ROOT = Path("data")

# Close any NetCDF datasets left open from a previous run in this kernel: on network filesystems
# (NFS), deleting a file that a process still has open leaves behind an undeletable ".nfs..." file
# and makes shutil.rmtree raise "Device or resource busy".
for _ds_name in ("ds_extUV", "ds_extVIS", "ds_bscUV", "ds_bscVIS", "ds_bscIR", "ds_HIRAC"):
    _ds_obj = globals().get(_ds_name)
    if _ds_obj is not None:
        try:
            _ds_obj.close()
        except Exception:
            pass

if CLEAN_DATA_FOLDER and DATA_ROOT.exists():
    shutil.rmtree(DATA_ROOT, ignore_errors=True)
    if DATA_ROOT.exists() and any(DATA_ROOT.iterdir()):
        print("Note: some leftover files in 'data' could not be removed (still in use elsewhere); "
              "continuing anyway - they won't interfere with this run's own subfolder.")

DATA_ROOT.mkdir(exist_ok=True)

# Everything produced by this run (downloaded files, extracted archives, plots) goes into its own
# subfolder named after station + time window, so different runs don't mix their files together.
run_id = f"{STATION}_{FROM_DATE.replace('-', '')}{FROM_TIME.replace(':', '')}_{TO_DATE.replace('-', '')}{TO_TIME.replace(':', '')}"
outdir = DATA_ROOT / run_id
import shutil

outdir.mkdir(parents=True, exist_ok=True)
print(f"Output folder for this run: {outdir}")


# Figures are shown inline and kept in memory. Nothing is written to disk while plotting:
# the export cell at the end of the notebook writes out whatever you decide to keep, so the
# choice does not have to be made before drawing anything.
DRAWN_FIGURES = {}


def register_figure(fig, stem):
    """Remember a figure under a name, for the export cell at the end."""
    DRAWN_FIGURES[stem] = fig


def sci_xaxis(ax, nticks=4):
    """Scientific notation on the x axis, with few ticks.

    Extinction and backscatter values are of the order of 1e-5 and 1e-7: written out in full
    the tick labels are long enough to overlap each other. Scientific notation moves the
    exponent to a single multiplier next to the axis.
    """
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nticks))
    ax.ticklabel_format(axis="x", style="sci", scilimits=(0, 0), useMathText=True)


def fold_x_offsets(fig):
    """Move the x10^n multiplier from the corner of the axis into the axis label.

    Matplotlib places that multiplier just below the right end of the axis, where it sits on
    top of a long axis label. Folding it into the label keeps both readable. The figure has
    to be drawn first, because the multiplier does not exist until then.
    """
    fig.canvas.draw()
    for ax in fig.axes:
        offset = ax.xaxis.get_offset_text().get_text()
        if offset:
            ax.xaxis.offsetText.set_visible(False)
            ax.set_xlabel(f"{ax.get_xlabel()} {offset}")

# Archives produced by the export cell at the end go here. The folder is emptied together with
# the rest, so the zips of previous runs do not pile up next to the notebook.
EXPORT_DIR = Path("exports")
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR, ignore_errors=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


# --- Settings and helpers shared by the in-situ sections at the end of the notebook -----
# Imported here rather than in the sections themselves: those are optional, and the NRT one
# was borrowing these from the Level 2 cell, so skipping that broke it.
import importlib
import importlib.util
import subprocess
import sys


FACILITIES_URL = "https://prod-actris-md2.nilu.no/facilities"
_insitu_nearby = []
NEARBY_RADIUS_KM = 30
# How many in-situ datasets may be opened at once. A facility's record can run to a decade of
# hourly data, and opening all of it is what fills the container's memory and kills the kernel.
MAX_INSITU_DATASETS = 6

# --- Helpers ----------------------------------------------------------------------------
# They live here, in a cell every run executes, rather than next to the sections that use
# them: those are optional, and someone jumping straight to the NRT part would otherwise
# find these undefined.

def _haversine_km(lat1, lon1, lat2, lon2):
    """Great-circle distance in kilometres."""
    radius = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dp = math.radians(lat2 - lat1)
    dl = math.radians(lon2 - lon1)
    a = math.sin(dp / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dl / 2) ** 2
    return 2 * radius * math.asin(math.sqrt(a))


def _time_coordinate(ds):
    """Name of the time coordinate: 'time' when present, otherwise the first datetime one."""
    if "time" in ds.coords or "time" in ds.dims:
        return "time"
    for name, coord in ds.coords.items():
        if np.issubdtype(coord.dtype, np.datetime64):
            return name
    return None


def _time_series(window, time_name, max_lines=4):
    """(label, series) pairs ready to plot, from one- or many-dimensional variables.

    An absorption product carries one value per wavelength, so its variables are
    (time, wavelength) rather than (time,): requiring one dimension found nothing to plot
    in files that were full of data.
    """
    series = []
    for name, variable in window.data_vars.items():
        if time_name not in variable.dims:
            continue
        others = [d for d in variable.dims if d != time_name]
        if not others:
            if np.isfinite(variable).sum():
                series.append((name, variable))
        else:
            # Take a few slices along the extra dimension, labelled by its coordinate
            # (the wavelength, typically) rather than by its index.
            extra = others[0]
            labels = (window[extra].values if extra in window.coords
                      else range(window.sizes[extra]))
            for index, label in enumerate(labels):
                if index >= 3:
                    break
                slice_ = variable.isel({extra: index})
                if np.isfinite(slice_).sum():
                    series.append((f"{name} @{label}", slice_))
        if len(series) >= max_lines:
            break
    return series[:max_lines]


def _facility_field(record, *names, default=""):
    """First of these keys the record actually has - the archive is not uniform."""
    for name in names:
        if record.get(name) not in (None, ""):
            return record[name]
    return default


### What to analyse

This notebook can do two things, and the second is a subset of the first.

**Optical products and aerosol typing** is the full run: it downloads the NetCDF files of the
measurement and plots extinction, backscatter, lidar ratio and depolarisation, then goes on to the
three typing algorithms. Use it to look at a measurement in full.

**Aerosol typing only** skips the optical part altogether — nothing is downloaded for it and the
plots are left out — and goes straight from the measurement to HIRAC, MAC and HETEAC-Flex, ending
with the three side by side. Noticeably faster, and the right choice when the question is which
aerosol was there rather than what the lidar measured.

The search runs either way: the typing products are requested per measurement, and that is where
the measurement comes from.

In [ ]:
# This notebook imports the widget classes by name rather than the module, so Dropdown and
# friends are used directly here, as in the other selection cells.
mode_dd = Dropdown(
    options=[("Optical products and aerosol typing (full run)", False),
             ("Aerosol typing only (faster, no optical downloads)", True)],
    value=globals().get("TYPING_ONLY", False),
    description="Analyse:", style={"description_width": "initial"},
    layout=Layout(width="520px"))
confirm_mode_btn = Button(description="Confirm", button_style="success", icon="check")
mode_out = Output()


def _on_mode_confirm(_):
    global TYPING_ONLY
    TYPING_ONLY = mode_dd.value
    with mode_out:
        mode_out.clear_output()
        if TYPING_ONLY:
            print("Aerosol typing only: the optical sections below will stay empty,")
            print("and nothing is downloaded for them.")
        else:
            print("Full run: optical products and aerosol typing.")


confirm_mode_btn.on_click(_on_mode_confirm)
display(VBox([mode_dd, confirm_mode_btn, mode_out]))

with mode_out:
    print(f"Not confirmed yet - will run as: "
          f"{'aerosol typing only' if TYPING_ONLY else 'optical products and aerosol typing'}.")

### Search ACTRIS Aerosol REmote Sensing (ARES) optical products

We use the ACTRIS/EARLINET REST API to search for optical products matching the station and time window above.
For more information, see <https://api.actris-ares.eu/api/swagger-ui/>.

In [ ]:
try:
    r = requests.get(
        "https://api.actris-ares.eu/api/services/restapi/opticalproducts/",
        params={
            "fromDate": FROM_DATE,
            "fromDayTime": FROM_TIME,
            "toDate": TO_DATE,
            "toDayTime": TO_TIME,
            "stations": STATION,
        },
    )
    r.raise_for_status()
    metadata = r.json()
    print(f"Data retrieved successfully! {len(metadata)} optical products found.")

except requests.exceptions.HTTPError as err:
    metadata = []
    if err.response.status_code == 404:
        print("No data available for the selected station in the specified time period.")
    else:
        print(f"An HTTP error occurred: {err}")

except Exception as e:
    metadata = []
    print(f"An unexpected error occurred: {e}")


### Download the NetCDF files

We extract, for every product found, its ID, filename and measurement ID from the metadata,
then download the corresponding NetCDF file.

In [ ]:
# The optical NetCDF files are only needed for the optical plots; in typing-only mode the
# measurement identifiers are still collected, since the typing requests are built from them.
ids_filenames = {}
ids_measurementids = {}
for pr in metadata:
    ids_filenames[pr.get("ID")] = pr.get("Filename")
    ids_measurementids[pr.get("ID")] = pr.get("Measurement Id")

if TYPING_ONLY:
    print(f"{len(ids_filenames)} product(s) found; download skipped (TYPING_ONLY).")
else:
    progress = IntProgress(min=0, max=len(ids_filenames), description="Download:")
    display(progress)
    for key, value in ids_filenames.items():
        get_request = "https://api.actris-ares.eu/api/services/restapi/opticalproducts/id/{id}/downloads"
        get_request = get_request.replace("{id}", str(key))
        resp = requests.get(get_request)
        resp.raise_for_status()
        (outdir / value).write_bytes(resp.content)
        progress.value += 1

    if ids_filenames:
        display(HTML(f"""
        <div style="padding: 10px; background-color: #96D7C6; color: #3E3E42; border-radius: 5px; font-weight: bold; text-align: center; border: 1px solid #3B4872;">
            All {len(ids_filenames)} products have been successfully downloaded!
        </div>
        """))


### Select the measurement to analyse

The search may return products belonging to several distinct measurements. Pick the one you want to
analyse from the dropdown below and click **"Confirm measurement"**.

Each entry shows the `measurement ID` followed by the optical products available for it, so you can
tell at a glance which measurement has the wavelengths you need. If you don't confirm anything, the
notebook simply uses the **first measurement in the list** and says so, rather than stopping with an
error. As with the other selectors, this cell is safe to re-run: an already-confirmed measurement is
preserved.

In [ ]:
FNAME_RE = re.compile(r"_([eb])(\d{3,4})_")

def parse_type_wavelength(filename):
    """Return (type, wavelength_nm) parsed from an EARLINET optical-product filename, or (None, None)."""
    m = FNAME_RE.search(filename)
    if not m:
        return None, None
    kind, wl = m.group(1), m.group(2)
    # Wavelength codes are zero-padded (0355, 0532, 1064), so a plain int() conversion is correct:
    # do NOT slice the last 3 digits, that would turn 1064 into 64.
    wl_nm = int(wl)
    if not 250 <= wl_nm <= 2100:
        print(f"WARNING: implausible wavelength {wl_nm} nm parsed from '{filename}'. "
              f"If you see values like 64 nm, the kernel is running an outdated version of this "
              f"notebook - restart the kernel and run all cells from the top.")
    return kind, wl_nm

def product_label(kind, wl):
    return f"{'ext' if kind == 'e' else 'bsc'}@{wl}nm"

# Group product IDs by measurement
measurements = {}
for pid, mid in ids_measurementids.items():
    measurements.setdefault(mid, []).append(pid)

if not measurements:
    raise RuntimeError("No measurements found for the selected station/date range. "
                        "Try widening the date range or changing the station code.")

# Build a descriptive label for each measurement: id + list of available products
measurement_options = []
for mid in sorted(measurements):
    prods = []
    for pid in measurements[mid]:
        kind, wl = parse_type_wavelength(ids_filenames[pid])
        if kind is not None:
            prods.append(product_label(kind, wl))
    prods_str = ", ".join(sorted(set(prods))) if prods else "no recognised products"
    measurement_options.append((f"{mid}  -  [{prods_str}]", mid))

print(f"Found {len(measurement_options)} distinct measurement(s).")

# Preserve an already-confirmed measurement if this cell gets re-run
if "SELECTED_MEASUREMENT_ID" not in globals():
    SELECTED_MEASUREMENT_ID = None
_valid_ids = [v for _, v in measurement_options]
_initial_mid = SELECTED_MEASUREMENT_ID if SELECTED_MEASUREMENT_ID in _valid_ids else _valid_ids[0]

measurement_dropdown = Dropdown(
    options=measurement_options,
    value=_initial_mid,
    description="Measurement:",
    style={"description_width": "initial"},
    layout=Layout(width="900px"),
)
confirm_meas_button = Button(description="Confirm measurement", button_style="success", icon="check")
confirm_meas_output = Output()

def _on_confirm_meas(b):
    global SELECTED_MEASUREMENT_ID
    SELECTED_MEASUREMENT_ID = measurement_dropdown.value
    with confirm_meas_output:
        confirm_meas_output.clear_output()
        print(f"Confirmed measurement: {SELECTED_MEASUREMENT_ID}")

confirm_meas_button.on_click(_on_confirm_meas)

display(measurement_dropdown, confirm_meas_button, confirm_meas_output)

with confirm_meas_output:
    if SELECTED_MEASUREMENT_ID:
        print(f"Confirmed measurement: {SELECTED_MEASUREMENT_ID} (already confirmed - you can move on)")
    else:
        print(f"Not confirmed yet - will default to the first measurement in the list: {_valid_ids[0]}")


### Resolve the filenames of the selected measurement

Filenames are parsed with the standard EARLINET naming convention:

`EARLINET_AerRemSen_{station}_Lev01_{type}{wavelength}_..._v{version}_qc{qc}.nc`

where `type` is `e` (extinction) or `b` (backscatter) and `wavelength` is a 4-digit code
(e.g. `0355`, `0532`, `1064`), so nothing has to be typed by hand.

In [ ]:
# If no measurement was confirmed above, fall back to the first one in the list so the notebook
# keeps running instead of erroring out.
SELECTED_MEASUREMENT_ID = globals().get("SELECTED_MEASUREMENT_ID")
if SELECTED_MEASUREMENT_ID not in measurements:
    SELECTED_MEASUREMENT_ID = sorted(measurements)[0]
    print(f"No measurement confirmed - defaulting to the first one: {SELECTED_MEASUREMENT_ID}")

selected_measurement_id = SELECTED_MEASUREMENT_ID
print(f"Selected measurement Id: {selected_measurement_id}")

# Build a lookup: (type, wavelength_nm) -> filename, for the selected measurement
files_by_type_wl = {}
for pid in measurements[selected_measurement_id]:
    fname = ids_filenames[pid]
    kind, wl = parse_type_wavelength(fname)
    if kind is not None:
        files_by_type_wl[(kind, wl)] = fname

print("Available optical products for this measurement:")
for (kind, wl), fname in sorted(files_by_type_wl.items()):
    label = "extinction" if kind == "e" else "backscatter"
    print(f"  {label} @{wl} nm -> {fname}")


## Optical products

Extinction, backscatter, lidar ratio and particle depolarisation for the measurement selected
above.

In **aerosol typing only** mode nothing is downloaded for this part and the cells below report that
they were skipped. Click the arrow next to this heading to fold the whole section away — JupyterLab
folds everything up to the next heading of the same level, which is *Aerosol type identification*.

### Select which products to plot

By default every optical product available for the selected measurement is plotted. If you only want
some of them, select the ones you need below (ctrl-click / cmd-click for multiple) and click
**"Confirm products"**. The plots further down will only show the confirmed selection.

In [ ]:
if TYPING_ONLY:
    # Downstream cells all ask whether a product was selected, so an empty selection is
    # enough to switch the optical sections off without touching them.
    SELECTED_PRODUCTS = ()
    print("TYPING_ONLY is on: no optical product selected, the plots below stay empty.")
else:
    available_products = sorted(files_by_type_wl.keys(), key=lambda t: (t[0], t[1]))
    product_options = [
        (f"{'Extinction' if kind == 'e' else 'Backscatter'} @{wl} nm", (kind, wl))
        for kind, wl in available_products
    ]

    # Preserve an already-confirmed choice if this cell gets re-run; default = everything available
    if "SELECTED_PRODUCTS" not in globals():
        SELECTED_PRODUCTS = None
    _initial_products = (
        tuple(p for p in SELECTED_PRODUCTS if p in available_products)
        if SELECTED_PRODUCTS else tuple(available_products)
    )

    product_select = SelectMultiple(
        options=product_options,
        value=_initial_products,
        description="Products:",
        style={"description_width": "initial"},
        rows=min(8, max(3, len(product_options))),
        layout=Layout(width="500px"),
    )
    confirm_prod_button = Button(description="Confirm products", button_style="success", icon="check")
    confirm_prod_output = Output()

    def _on_confirm_prod(b):
        global SELECTED_PRODUCTS
        SELECTED_PRODUCTS = list(product_select.value)
        with confirm_prod_output:
            confirm_prod_output.clear_output()
            labels = [f"{'ext' if k == 'e' else 'bsc'}@{w}nm" for k, w in SELECTED_PRODUCTS]
            print(f"Confirmed products: {', '.join(labels) if labels else '(none selected!)'}")

    confirm_prod_button.on_click(_on_confirm_prod)

    display(product_select, confirm_prod_button, confirm_prod_output)

    # If nothing was ever confirmed, fall back to plotting everything available
    if SELECTED_PRODUCTS is None:
        SELECTED_PRODUCTS = list(available_products)
        with confirm_prod_output:
            print("Defaulting to all available products (click the button to restrict the selection).")
    else:
        with confirm_prod_output:
            labels = [f"{'ext' if k == 'e' else 'bsc'}@{w}nm" for k, w in SELECTED_PRODUCTS]
            print(f"Confirmed products: {', '.join(labels)} (already confirmed - you can move on)")


### Open the datasets

We open with `xarray` the products that are both available for the selected measurement and included
in the product selection confirmed above.

In [ ]:
def open_if_available(kind, wl):
    """Open the dataset for (kind, wavelength) only if it exists AND was selected above."""
    if (kind, wl) not in SELECTED_PRODUCTS:
        return None
    fname = files_by_type_wl.get((kind, wl))
    if fname is None:
        return None
    return xr.open_dataset(outdir / fname)


def open_backscatter(wl):
    """Backscatter at a given wavelength, from the 'b' file when there is one.

    Raman extinction files ('e') also carry a backscatter profile at the same wavelength.
    When a measurement has no elastic 'b' product, that profile is the only backscatter
    available, and ignoring it would leave the plot empty for no good reason. The 'b' file
    is preferred when both exist, since that is the dedicated product.

    Returns (dataset, source) where source is "b" or "e", or (None, None).
    """
    if ("b", wl) in SELECTED_PRODUCTS and files_by_type_wl.get(("b", wl)):
        return xr.open_dataset(outdir / files_by_type_wl[("b", wl)]), "b"

    # Fall back to the Raman file, but only if it really contains a backscatter variable
    if ("e", wl) in SELECTED_PRODUCTS and files_by_type_wl.get(("e", wl)):
        ds = xr.open_dataset(outdir / files_by_type_wl[("e", wl)])
        if "backscatter" in ds.data_vars:
            return ds, "e"
        ds.close()
    return None, None


# Every optical product is handled at all three wavelengths. Nothing is assumed to exist:
# a wavelength simply does not appear in the plots when the station did not measure it.
# Extinction at 1064 nm, for instance, exists only at stations with a rotational Raman
# channel, and depolarisation is not available at every wavelength either.
WAVELENGTHS = (355, 532, 1064)
WL_COLOURS = {355: "blue", 532: "green", 1064: "red"}

ds_ext = {}
for _wl in WAVELENGTHS:
    _d = open_if_available("e", _wl)
    if _d is not None:
        ds_ext[_wl] = _d

ds_bsc, bsc_source = {}, {}
for _wl in WAVELENGTHS:
    _d, _src = open_backscatter(_wl)
    if _d is not None:
        ds_bsc[_wl] = _d
        bsc_source[_wl] = _src

_opened = ([f"ext@{wl}" for wl in sorted(ds_ext)] + [f"bsc@{wl}" for wl in sorted(ds_bsc)])
if TYPING_ONLY:
    print("Skipped: aerosol typing only.")
else:
    print("Datasets opened for plotting:", ", ".join(_opened) if _opened else "none")

_from_raman = [f"{wl} nm" for wl in sorted(ds_bsc) if bsc_source[wl] == "e"]
if _from_raman:
    print(f"Backscatter at {', '.join(_from_raman)} taken from the Raman extinction file: "
          f"no dedicated backscatter ('b') product exists for this measurement.")

### Plot: Aerosol Particle Light Extinction Coefficient

In [ ]:
panels = [(wl, ds_ext[wl]) for wl in sorted(ds_ext)]

if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
    if len(panels) == 1:
        axes = [axes]
    for ax, (wl, ds) in zip(axes, panels):
        extinction = ds.extinction.fillna(0)
        ax.plot(extinction[0][0][:], ds.altitude[:], color=WL_COLOURS.get(wl, "black"))
        ax.set_title(f"Aerosol Particle Light Extinction Coefficient @{wl} nm")
        ax.set_xlabel("Aerosol Particle Light Extinction Coefficient (1/m)")
        ax.set_ylabel("Altitude (m asl)")
        ax.set_xlim(-0.00001, 0.0002)
        sci_xaxis(ax)
    fold_x_offsets(fig)
    plt.tight_layout()
    register_figure(fig, f"{STATION}_extinction")
    plt.show()
else:
    print("Skipped: aerosol typing only." if TYPING_ONLY
          else "No extinction products available for this measurement.")

### Plot: Aerosol Particle Backscatter Coefficient

Backscatter is taken from the dedicated elastic product (`b` files) whenever the measurement has
one. Raman extinction files (`e`) also contain a backscatter profile at the same wavelength, so when
no `b` product exists for a wavelength that profile is used instead — the plot title states when
this happens, and a note is printed when the datasets are opened.

In [ ]:
panels = [(wl, ds_bsc[wl], bsc_source[wl]) for wl in sorted(ds_bsc)]

if panels:
    fig, axes = plt.subplots(1, len(panels), figsize=(5 * len(panels), 5))
    if len(panels) == 1:
        axes = [axes]
    for ax, (wl, ds, source) in zip(axes, panels):
        backscatter = ds.backscatter.fillna(0)
        ax.plot(backscatter[0][0][:], ds.altitude[:], color=WL_COLOURS.get(wl, "black"))
        # The title says where the profile came from, so a Raman-derived backscatter is never
        # mistaken for the dedicated elastic product.
        origin = "" if source == "b" else "\n(from the Raman extinction file)"
        ax.set_title(f"Aerosol Particle Backscatter Coefficient @{wl} nm{origin}")
        ax.set_xlabel("Aerosol Particle Backscatter Coefficient (1/(m*sr))")
        ax.set_ylabel("Altitude (m asl)")
        ax.set_xlim(-0.0000001, 0.000003)
        sci_xaxis(ax)
    fold_x_offsets(fig)
    plt.tight_layout()
    register_figure(fig, f"{STATION}_backscatter")
    plt.show()
else:
    print("Skipped: aerosol typing only." if TYPING_ONLY
          else "No backscatter products available for this measurement.")

### Plot: Lidar Ratio and Particle Depolarization

In [ ]:
lidar_panels, depol_panels = [], []

for wl in sorted(ds_ext):
    if "lidarratio" in ds_ext[wl].data_vars:
        lidar_panels.append((f"Lidar Ratio @{wl} nm", ds_ext[wl].lidarratio.fillna(0),
                             ds_ext[wl].altitude, WL_COLOURS.get(wl, "black"),
                             "Lidar Ratio (sr)", (-10, 200)))

for wl in sorted(ds_bsc):
    if "particledepolarization" in ds_bsc[wl].data_vars:
        depol_panels.append((f"Particle Depolarization @{wl} nm",
                             ds_bsc[wl].particledepolarization.fillna(0), ds_bsc[wl].altitude,
                             "orange", "Particle Depolarization", (-0.1, 1.5)))

# One row per quantity rather than a single long strip: with three wavelengths each, five or
# six panels side by side become too narrow to read.
rows = [r for r in (lidar_panels, depol_panels) if r]

if rows:
    ncols = max(len(r) for r in rows)
    fig, axes = plt.subplots(len(rows), ncols, figsize=(5.5 * ncols, 5.5 * len(rows)),
                             squeeze=False)
    for i, row in enumerate(rows):
        for j in range(ncols):
            ax = axes[i][j]
            if j >= len(row):
                ax.set_visible(False)      # ragged row: hide the leftover axes
                continue
            title, values, altitude, color, xlabel, xlim = row[j]
            ax.plot(values[0][0][:], altitude[:], color=color)
            ax.set_title(title)
            ax.set_xlabel(xlabel)
            ax.set_ylabel("Altitude (m asl)")
            ax.set_xlim(*xlim)
    plt.tight_layout()
    register_figure(fig, f"{STATION}_lidarratio_depolarization")
    plt.show()
else:
    print("Skipped: aerosol typing only." if TYPING_ONLY
          else "No lidar ratio / depolarization products available for this measurement.")

# Depolarisation lives in the elastic ('b') product. When the backscatter had to be taken from
# a Raman file, its absence says nothing about whether the station measures it - worth stating,
# so a missing panel is not read as "this station has no depolarisation channel".
_no_depol_raman = [wl for wl in sorted(ds_bsc)
                   if bsc_source[wl] == "e" and "particledepolarization" not in ds_bsc[wl].data_vars]
if _no_depol_raman:
    print(f"\nNo depolarization at {', '.join(f'{wl} nm' for wl in _no_depol_raman)}: the "
          f"backscatter there comes from the Raman file, which does not carry that variable. "
          f"A dedicated 'b' product would be needed.")

_missing = [wl for wl in WAVELENGTHS if wl in ds_bsc
            and "particledepolarization" not in ds_bsc[wl].data_vars
            and wl not in _no_depol_raman]
if _missing:
    print(f"No depolarization variable in the backscatter product at "
          f"{', '.join(f'{wl} nm' for wl in _missing)}.")

### Discussion

The last plot boxes show intensive properties not depending on teh aerosol quantity but only on their chemical/physical properties, i.e. strognly depends on the aerosol types. 
here 2 main aerosol layers are identified: between 1.5 and 4.5 km asl amnd between 7 and 9 km asl. They have different lidar ratio and depolarization, which means different aerosol types. 
In the following, aerosol typign products are exploitaing as tool for identigying the fire plume.

## Aerosol type identification

The ACTRIS/EARLINET REST API also provides access to aerosol typing products (HIRAC, MAC and HETEAC-Flex).
More info: <https://earlinet.eu/our-knowledge/access-to-data/>

We use the `measurement ID` of the measurement selected above (`selected_measurement_id`) — no
manual lookup or hardcoded ID is needed.

In [ ]:
def download_product(kind):
    """Download one typing product archive, returning its filename or None.

    The endpoint answers 200 with an error page or an empty body when a product does not
    exist for the measurement, so the reply is checked before it is written: otherwise the
    problem only surfaces several cells later as "BadZipFile: File is not a zip file".
    """
    path = f"{selected_measurement_id}_{kind}.zip"
    try:
        r = requests.get(
            "https://api.actris-ares.eu/api/services/restapi/products/downloads",
            params={"kind": kind, "measurementId": selected_measurement_id},
            timeout=120,
        )
    except Exception as exc:  # noqa: BLE001 - a network failure must be readable
        print(f"{kind}: download failed ({type(exc).__name__}: {exc})")
        return None

    if r.status_code != 200:
        print(f"{kind}: the server answered {r.status_code}, no archive downloaded.")
        return None
    if not r.content:
        print(f"{kind}: the server returned an empty response - "
              f"this product is probably not available for this measurement.")
        return None
    if r.content[:2] != b"PK":                    # every zip starts with these two bytes
        preview = r.content[:200].decode("utf-8", "replace").strip().replace("\n", " ")
        print(f"{kind}: the reply is not a zip archive - this product is probably not "
              f"available for this measurement.")
        print(f"   the server said: {preview[:160]}")
        return None

    (outdir / path).write_bytes(r.content)
    # MAC and HETEAC are layer products of a few tens of kB, so a fixed MB unit would show
    # every one of them as "0.0 MB".
    size = len(r.content)
    shown = f"{size / 1e6:.1f} MB" if size >= 1e6 else f"{size / 1e3:.0f} kB"
    print(f"{kind}: {shown} downloaded")
    return path


productKind = "MAC"
productKind2 = "HIRAC"
productKind3 = "HETEAC"

# Downloaded and reported in the order the sections below use them
file_path2 = download_product(productKind2)   # HIRAC
file_path = download_product(productKind)     # MAC
file_path3 = download_product(productKind3)   # HETEAC-Flex

_products = (("HIRAC", file_path2), ("MAC", file_path), ("HETEAC", file_path3))
_got = [k for k, p in _products if p]
_missing = [k for k, p in _products if not p]
print(f"\nAvailable for this measurement: {', '.join(_got) if _got else 'none'}")
if _missing:
    print(f"Not available: {', '.join(_missing)} - the corresponding sections below will say so "
          f"and be skipped. This is normal: not every measurement has every typing product.")

### Extract the HIRAC archive

In [ ]:
if not file_path2:
    raise RuntimeError(
        "No HIRAC archive for this measurement, so the aerosol typing plots below cannot be "
        "produced. The MAC and HETEAC-Flex sections further down may still work."
    )

with zipfile.ZipFile(outdir / file_path2, "r") as zip_ref:
    zip_ref.extractall(outdir)
    extracted_names = zip_ref.namelist()

print("Extracted files:", extracted_names)

### Automatically find the HIRAC NetCDF file

Instead of typing the HIRAC filename by hand, we search the `data` folder for the extracted
`.nc` file containing "HIRAC" in its name.

In [ ]:
hirac_candidates = sorted(glob.glob(str(outdir / "**" / "*HIRAC*.nc"), recursive=True))
if not hirac_candidates:
    raise FileNotFoundError("No HIRAC .nc file found after extracting the archive.")

filename_HIRAC = Path(hirac_candidates[0]).name
print(f"Using HIRAC file: {filename_HIRAC}")

# Station location label for the plot title, taken from the station list fetched earlier
location = next(
    (f"{s.get('Location')}" for s in stations_list if s.get("ID") == STATION),
    STATION,
)


### Plot: HIRAC aerosol classification

In [ ]:
# Classes and colours of the HIRAC scheme, defined here rather than inside the try block so
# that the side-by-side comparison further down draws the same legend.
HIRAC_CLASSES = ["Clean", "No\nType", "Small", "Large\nspherical", "Mixture", "Large\naspherical"]
HIRAC_COLORS = ("#c7d8e5", "#999999", "#ddcc77", "#56b1f7", "#b44c1c", "#7f2f00")

# The plot is wrapped so that a missing file or an unexpected variable is reported in words
# rather than only as a traceback: these products are not published for every measurement.
try:
    ds_HIRAC = xr.open_dataset(Path(hirac_candidates[0]))
    type_mask = ds_HIRAC.aerosol_classification.fillna(0)
    height_f = ds_HIRAC.altitude
    time_f = ds_HIRAC.time

    name_f = filename_HIRAC.replace(".nc", "")

    plt.rc("font", family="serif")
    plt.rcParams.update({"font.size": 18})

    # Define categories
    cmap = mcolors.ListedColormap(HIRAC_COLORS)
    bounds = [0, 1, 2, 3, 4, 5, 6]
    norm = mcolors.BoundaryNorm(bounds, cmap.N)

    fig = plt.figure(figsize=(10.0, 5.0))
    ax = fig.add_axes([0.07, 0.13, 0.99, 0.68])

    xx_, yy_ = np.meshgrid(time_f, height_f / 1000.0)
    cs1 = ax.pcolormesh(xx_, yy_, type_mask.transpose(), cmap=cmap, norm=norm)

    ax.set_ylim(0, 12)
    ax.set(ylabel="Altitude [km asl]")
    ax.set(xlabel="Time UTC [HH:MM]")

    locator = mdates.AutoDateLocator()
    formatter = mdates.ConciseDateFormatter(locator)
    ax.xaxis.set_major_locator(locator)
    ax.xaxis.set_major_formatter(formatter)

    ax.yaxis.set_minor_locator(MultipleLocator(1))
    ax.tick_params(which="both", top=True, right=True, direction="in")
    ax.set_title(f"Aerosol classification: {location}\n{name_f}", fontsize=18)

    divider = make_axes_locatable(ax)
    cax = divider.append_axes("right", size="4%", pad=0.05)
    cbar = plt.colorbar(
        cs1, cax=cax,
        format=lambda x, _: {0.5: "Clean", 1.5: "No\nType", 2.5: "Small", 3.5: "Large\nspherical",
                              4.5: "Mixture", 5.5: "Large\naspherical"}[x],
    )
    cbar.set_ticks([0.5, 1.5, 2.5, 3.5, 4.5, 5.5])
    cbar.ax.set_yticklabels(
        ["Clean", "No\nType", "Small", "Large\nspherical", "Mixture", "Large\naspherical"],
        rotation="horizontal", fontsize=12, va="center",
    )

    register_figure(fig, name_f)

    plt.show()
except (IndexError, FileNotFoundError):
    print("No HIRAC file to plot: the archive was not downloaded, or held no NetCDF file.")
    print("See the download section above - HIRAC is not published for every measurement.")
except (KeyError, AttributeError) as exc:
    print(f"The HIRAC file does not contain what this plot expects ({exc}).")
    print("The product format may have changed; the variables found in the file are listed by:")
    print("    import xarray as xr; print(xr.open_dataset(hirac_candidates[0]))")
except Exception as exc:  # noqa: BLE001 - anything else, but say what happened
    import traceback
    print(f"The HIRAC plot could not be produced ({type(exc).__name__}: {exc}).")
    print("Full details below.")
    traceback.print_exc()

### HIRAC discussion

Layers of different kinds are observed, with the lofted layer (around 8 km) clearly related to the
fire plume (small particles).

## MAC aerosol typing: layer classification

The MAC archive was downloaded together with HIRAC earlier in the notebook, but until now only
HIRAC was used. MAC describes the atmosphere as a set of **discrete layers**, each with a base, a
top and an aerosol classification — a complementary view to the continuous HIRAC field: fewer
details, but an explicit answer to "which layers are there, and what are they made of".

Each layer is drawn as a rectangle spanning its base-to-top range, coloured by aerosol type.

In [ ]:
def extract_typing(archive_name, label, pattern, exclude=()):
    """Extract a typing archive and list its NetCDF files, or explain why there are none."""
    if not archive_name:
        print(f"No {label} archive for this measurement - nothing to plot in this section.")
        return []
    try:
        with zipfile.ZipFile(outdir / archive_name, "r") as zip_ref:
            zip_ref.extractall(outdir)
            print("Extracted:", zip_ref.namelist())
    except zipfile.BadZipFile:
        print(f"The {label} archive is not a readable zip file. It may have been truncated; "
              f"re-run the download cell above.")
        return []

    found = sorted(glob.glob(str(outdir / "**" / pattern), recursive=True))
    if not found and exclude:
        found = sorted(f for f in glob.glob(str(outdir / "**" / "*.nc"), recursive=True)
                       if not any(k in Path(f).name for k in exclude))
    if not found:
        present = [Path(f).name for f in glob.glob(str(outdir / "**" / "*.nc"), recursive=True)]
        print(f"No {label} NetCDF file after extracting the archive. Files present: {present}")
        return []

    print(f"\n{len(found)} {label} file(s):")
    for f in found:
        print("  ", Path(f).name)
    return found


mac_files = extract_typing(file_path, "MAC", "*MAC*.nc",
                           exclude=("HIRAC", "HETEAC", "AerRemSen"))

### Plotting functions

Adapted from `plot_mac_column.py`. The colour of each aerosol class can be changed in
`AEROSOL_TYPES` below.

In [ ]:
from datetime import datetime

from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.ticker import AutoMinorLocator

# Classification tables. MAC and HETEAC-Flex use different schemes and a different variable
# name, but the layer geometry is the same, so the plotting code below is shared.
MAC_TYPES = {
    0: ("Not classified", "#bababa"),
    1: ("Clean Continental", "#abd9e9"),
    2: ("Smoke", "#6a51a3"),
    3: ("Dust", "#d7301f"),
    4: ("Marine", "#4575b4"),
    5: ("Polluted Continental", "#cbc9e2"),
    6: ("Polluted Dust", "#fc8d59"),
    7: ("Mixed Dust", "#fdcc8a"),
}

HETEAC_TYPES = {
    1: ("CS", "#abd9e9"),
    2: ("FSA", "#6a51a3"),
    3: ("FSNA", "#9e9ac8"),
    4: ("CNS", "#d7301f"),
    5: ("CNS/CS", "#fc8d59"),
    6: ("CNS/FSA", "#fdcc8a"),
    7: ("CNS/FSNA", "#fef0d9"),
    8: ("FSA/FSNA", "#cbc9e2"),
    9: ("FSNA/CS", "#f2f0f7"),
    10: ("Not-typed", "#bababa"),
}

_KM_UNITS = {"km", "kilometre", "kilometres", "kilometer", "kilometers"}


def _parse_typing_time(value):
    """Parse '2024-08-12T00:00:00Z' as written in the global attributes."""
    return datetime.strptime(value, "%Y-%m-%dT%H:%M:%SZ")


def _height_to_metres_factor(var):
    """Factor converting this variable's values to metres.

    The original scripts assumed base/top were in metres, but the products store them in
    kilometres (`units = "km"`), which produced an axis labelled "m" with values around 3.
    The unit is read from the variable attributes; if it is absent, the magnitude decides.
    """
    unit = str(var.attrs.get("units", "")).strip().lower()
    if unit in _KM_UNITS:
        return 1000.0
    if unit in {"m", "metre", "metres", "meter", "meters"}:
        return 1.0
    values = var.values
    guessed_km = len(values) > 0 and float(max(values)) < 50
    print(f"NOTE: no usable 'units' attribute on '{var.name}' (found {unit!r}); "
          f"assuming {'kilometres' if guessed_km else 'metres'} from the value range.")
    return 1000.0 if guessed_km else 1.0


def load_typing_layers(nc_path, type_var):
    """One dict per detected layer of a typing file, heights normalised to metres.

    `type_var` is the classification variable: 'aerosol_classification' for MAC,
    'aerosol_type' for HETEAC-Flex.
    """
    with xr.open_dataset(nc_path) as ds:
        t0 = _parse_typing_time(ds.attrs["measurement_start_datetime"])
        t1 = _parse_typing_time(ds.attrs["measurement_stop_datetime"])
        if type_var not in ds.variables:
            raise KeyError(f"'{type_var}' not found in {Path(nc_path).name}. "
                           f"Variables present: {list(ds.variables)}")
        to_m = _height_to_metres_factor(ds["base"])
        n = ds.sizes.get("layers", 0)
        return [
            dict(t0=t0, t1=t1,
                 base=float(ds["base"].values[i]) * to_m,
                 top=float(ds["top"].values[i]) * to_m,
                 aerosol_type=int(ds[type_var].values[i]))
            for i in range(n)
        ]


def _style_axes(ax, date_axis=False):
    ax.tick_params(which="both", bottom=True, top=True, left=True, right=True,
                   width=1, direction="in")
    if not date_axis:
        # AutoMinorLocator on a date axis adds clutter and fights the date locator
        ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.grid(linewidth=1, color="w")
    for side in ("top", "bottom", "left", "right"):
        ax.spines[side].set_linewidth(1)
    ax.set_facecolor("lightgray")


def _legend_for(layers, types, ax, **kw):
    present = sorted({l["aerosol_type"] for l in layers})
    handles = [
        Line2D([0], [0], marker="s", linestyle="", markersize=10,
               markerfacecolor=types.get(t, (f"type {t}", "black"))[1],
               markeredgecolor="k",
               label=types.get(t, (f"type {t}", "black"))[0])
        for t in present
    ]
    ax.legend(handles=handles, **kw)


def plot_typing_profile(nc_path, types, product, type_var, height_units="km",
                        column_width=0.8, ax=None):
    """One typing file as a single column of stacked, colour-coded layers."""
    layers = load_typing_layers(nc_path, type_var)
    if not layers:
        raise ValueError(f"No layers found in {Path(nc_path).name}")

    scale = 1.0 if height_units == "m" else 1000.0
    fig, ax = (ax.figure, ax) if ax is not None else plt.subplots(figsize=(4.2, 5.2))

    for lyr in layers:
        _, colour = types.get(lyr["aerosol_type"], ("unknown", "black"))
        ax.add_patch(Rectangle((-column_width / 2, lyr["base"] / scale), column_width,
                               (lyr["top"] - lyr["base"]) / scale,
                               facecolor=colour, edgecolor="k", linewidth=0.4))

    # Limits come from the data. The original scripts forced ax.set_ylim(0, 10), which only
    # makes sense in kilometres and hides everything when the units are metres.
    ymax = max(l["top"] for l in layers) / scale
    ax.set_xlim(-0.7, 0.7)
    ax.set_ylim(0, ymax * 1.15)
    ax.set_xticks([])

    t0, t1 = layers[0]["t0"], layers[0]["t1"]
    ax.set_ylabel(f"Height a.s.l. ({height_units})")
    ax.set_xlabel(t0.strftime("%Y-%m-%d %H:%M") + " - " + t1.strftime("%H:%M UTC"))
    where = location if location is not None else globals().get("location", "")
    ax.set_title(f"{product} classification\n{where}")
    _legend_for(layers, types, ax, prop={"size": 7})
    _style_axes(ax)
    fig.tight_layout()
    return fig, ax


def plot_typing_curtain(nc_paths, types, product, type_var, height_units="km",
                        mode="auto", column_width=0.8, gap_ratio_threshold=20, ax=None, location=None):
    """Several typing files side by side.

    mode "continuous"  - real time axis, for profiles close together in time
         "categorical" - one evenly spaced column per file, for sparse case studies
         "auto"        - categorical when the files are spread far apart in time
    """
    per_file = [(p, load_typing_layers(p, type_var)) for p in nc_paths]
    per_file = [pf for pf in per_file if pf[1]]
    if not per_file:
        raise ValueError("No layers found in the given files.")
    per_file.sort(key=lambda pf: pf[1][0]["t0"])

    all_layers = [l for _, layers in per_file for l in layers]
    scale = 1.0 if height_units == "m" else 1000.0

    if mode == "auto":
        durations = [(l["t1"] - l["t0"]).total_seconds() for l in all_layers]
        typical = max(sorted(durations)[len(durations) // 2], 60.0)
        span = (per_file[-1][1][0]["t0"] - per_file[0][1][0]["t0"]).total_seconds()
        mode = "categorical" if span > gap_ratio_threshold * typical else "continuous"
        print(f"mode='auto' resolved to '{mode}'")

    # An axis can be supplied, so that this product can be drawn next to the others in the
    # comparison figure instead of in a figure of its own.
    if ax is None:
        fig, ax = plt.subplots(figsize=(max(9, 1.4 * len(per_file)), 5))
    else:
        fig = ax.figure

    if mode == "continuous":
        for lyr in all_layers:
            _, colour = types.get(lyr["aerosol_type"], ("unknown", "black"))
            x0 = mdates.date2num(lyr["t0"])
            ax.add_patch(Rectangle((x0, lyr["base"] / scale),
                                   mdates.date2num(lyr["t1"]) - x0,
                                   (lyr["top"] - lyr["base"]) / scale,
                                   facecolor=colour, edgecolor="k", linewidth=0.4))
        tmin = min(mdates.date2num(l["t0"]) for l in all_layers)
        tmax = max(mdates.date2num(l["t1"]) for l in all_layers)
        pad = max((tmax - tmin) * 0.05, 1 / 24 / 60)
        ax.set_xlim(tmin - pad, tmax + pad)
        ax.xaxis_date()
        # A fixed formatter on a short interval produces many overlapping labels; letting the
        # locator choose the ticks and the formatter adapt to the span avoids that.
        locator = mdates.AutoDateLocator(minticks=3, maxticks=7)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        ax.set_xlabel("Time (UTC)")
    else:
        for i, (_, layers) in enumerate(per_file):
            for lyr in layers:
                _, colour = types.get(lyr["aerosol_type"], ("unknown", "black"))
                ax.add_patch(Rectangle((i - column_width / 2, lyr["base"] / scale),
                                       column_width,
                                       (lyr["top"] - lyr["base"]) / scale,
                                       facecolor=colour, edgecolor="k", linewidth=0.4))
        ax.set_xlim(-0.7, len(per_file) - 0.3)
        ax.set_xticks(range(len(per_file)))
        ax.set_xticklabels([l[0]["t0"].strftime("%Y-%m-%d\n%H:%M") for _, l in per_file])
        ax.set_xlabel("Measurement date/time (UTC) - columns not to time scale")

    ymax = max(l["top"] for l in all_layers) / scale
    ax.set_ylim(0, ymax * 1.15)
    ax.set_ylabel(f"Height a.s.l. ({height_units})")
    ax.set_title(f"{product} classification\n{location}")
    _legend_for(all_layers, types, ax, loc="upper left", bbox_to_anchor=(1.01, 1),
                borderaxespad=0, title="Aerosol type")
    _style_axes(ax, date_axis=(mode == "continuous"))
    fig.tight_layout()
    return fig, ax


def typing_plot_controls(files, types, product, type_var, prefix):
    """Show the layout/units/save controls for a typing product and draw the plot.

    Used for both MAC and HETEAC-Flex, so the two sections stay in step.
    """
    if not files:
        print(f"No {product} file for this measurement - nothing to plot here.")
        return

    several = len(files) > 1

    mode_dd = Dropdown(
        options=[("One plot per file", "per-file"),
                 ("All files, real time axis", "continuous"),
                 ("All files, one column each", "categorical"),
                 ("Choose automatically", "auto")],
        value="auto" if several else "per-file",
        description="Layout:", style={"description_width": "initial"},
        layout=Layout(width="420px"))
    # Heights are always shown in kilometres: that is the convention for these products,
    # and it is what the algorithm authors expect to see. The plotting functions still take a
    # height_units argument, so metres remain available when calling them directly.
    units = "km"
    btn = Button(description="Redraw", button_style="success", icon="refresh")
    out = Output()

    def draw(_=None):
        with out:
            out.clear_output(wait=True)
            mode = mode_dd.value
            try:
                # Saving follows the notebook-wide control in the "Saving the figures"
                # section, like every other figure here, rather than a separate switch.
                if mode == "per-file":
                    for path in files:
                        fig, _ax = plot_typing_profile(path, types, product, type_var,
                                                       height_units=units)
                        register_figure(fig, Path(path).stem)
                        plt.show()
                else:
                    fig, _ax = plot_typing_curtain(files, types, product, type_var,
                                                   height_units=units, mode=mode)
                    register_figure(fig, f"{prefix}_curtain")
                    plt.show()
            except (KeyError, ValueError) as exc:
                # xarray errors run to several lines of installation advice; the first line
                # is the part that says what actually went wrong.
                reason = str(exc).split("\n")[0][:160]
                print(f"The {product} file could not be read as expected: {reason}")
                print("The product format may have changed, or the file may be incomplete. "
                      "Re-running the download cell above is worth a try.")
            except FileNotFoundError:
                print(f"The {product} file is no longer on disk. The run folder is emptied at "
                      f"the start of each run - re-run the download and extraction cells above.")
            except Exception as exc:  # noqa: BLE001 - anything else, but say what happened
                import traceback
                print(f"The {product} plot could not be produced "
                      f"({type(exc).__name__}: {exc}). Full details below.")
                traceback.print_exc()

    # The dropdowns redraw on their own: changing a value and seeing nothing happen until a
    # button is pressed is a poor way to explore the data. The button stays as a refresh.
    mode_dd.observe(draw, names="value")
    btn.on_click(draw)

    controls = ([mode_dd] if several else []) + [btn, out]
    display(VBox(controls))
    if not several:
        print(f"One {product} file for this measurement: a single profile is drawn. The layout "
              f"options only apply when a measurement yields several files.")
    draw()

### Plot

Each measurement normally yields a single MAC file, so one profile is drawn: a column of layers,
coloured by aerosol type.

When a measurement yields several files, a **Layout** menu appears with three ways of arranging
them — one plot per file, all files on a real time axis, or one evenly spaced column per file —
plus an automatic choice based on how far apart the profiles are in time.

To keep any of these figures, use the [export cell](#export) at the end of the notebook.

In [ ]:
typing_plot_controls(mac_files, MAC_TYPES, "MAC", "aerosol_classification", prefix="MAC")

## HETEAC-Flex aerosol typing

HETEAC-Flex is a third typing product for the same measurement. Like MAC it describes the
atmosphere as discrete layers, but it uses a different classification scheme, built on combinations
of coarse/fine and spherical/non-spherical, absorbing/non-absorbing components. The meaning of each
code is given with the plot below.

The plotting controls are the same as for MAC.

In [ ]:
heteac_files = extract_typing(file_path3, "HETEAC-Flex", "*HETEAC*.nc")

### Plot

As for MAC, a single file gives one profile; the **Layout** menu appears only when a measurement
yields several. To keep the figure, use the [export cell](#export) at the end.

The legend uses the HETEAC-Flex codes:

| Code | Meaning | | Code | Meaning |
|---|---|---|---|---|
| CS | coarse spherical | | CNS/CS | coarse non-spherical / coarse spherical |
| FSA | fine spherical absorbing | | CNS/FSA | coarse non-spherical / fine spherical absorbing |
| FSNA | fine spherical non-absorbing | | CNS/FSNA | coarse non-spherical / fine spherical non-absorbing |
| CNS | coarse non-spherical | | FSA/FSNA, FSNA/CS | mixtures |
| Not-typed | no classification assigned | | | |

In [ ]:
typing_plot_controls(heteac_files, HETEAC_TYPES, "HETEAC-Flex", "aerosol_type", prefix="HETEAC")

## The three algorithms side by side

HIRAC, MAC and HETEAC-Flex classify the same measurement independently, and they do not always
agree. Reading them one after the other makes the differences hard to see, so this section puts
them on one figure, sharing the altitude axis.

They are not the same kind of product, and the figure keeps that visible rather than hiding it:
HIRAC resolves the classification in time and altitude, so it is drawn as a curtain, while MAC and
HETEAC-Flex describe discrete layers and are drawn as columns. Each keeps its own colour scheme,
because the classes are not the same either — a disagreement between two panels is a disagreement
between algorithms, not a difference in how they are drawn.

In [ ]:
def compare_typing(hirac_path=None, mac_paths=None, heteac_paths=None, top_km=None):
    """One figure with whichever of the three typing products is available."""
    panels = []
    if hirac_path:
        panels.append(("HIRAC", "grid", hirac_path))
    if mac_paths:
        panels.append(("MAC", "layers", mac_paths))
    if heteac_paths:
        panels.append(("HETEAC-Flex", "layers", heteac_paths))

    if not panels:
        print("None of the three typing products is available for this measurement.")
        return None

    missing = [name for name in ("HIRAC", "MAC", "HETEAC-Flex")
               if name not in [p[0] for p in panels]]
    if missing:
        print(f"Not available for this measurement: {', '.join(missing)}.")
        print("Not every product exists for every measurement - the rest are drawn below.")

    fig, axes = plt.subplots(1, len(panels), figsize=(6.0 * len(panels), 5.5), sharey=True)
    if len(panels) == 1:
        axes = [axes]

    for ax, (label, kind, source) in zip(axes, panels):
        try:
            if kind == "grid":
                with xr.open_dataset(source) as ds:
                    classification = ds.aerosol_classification.fillna(0)
                    grid_x, grid_y = np.meshgrid(ds.time, ds.altitude / 1000.0)
                    cmap = mcolors.ListedColormap(HIRAC_COLORS)
                    norm = mcolors.BoundaryNorm(list(range(len(HIRAC_COLORS) + 1)), cmap.N)
                    mesh = ax.pcolormesh(grid_x, grid_y, classification.transpose().values,
                                         cmap=cmap, norm=norm)
                    cbar = fig.colorbar(mesh, ax=ax, fraction=0.046, pad=0.02)
                    cbar.set_ticks([i + 0.5 for i in range(len(HIRAC_CLASSES))])
                    cbar.ax.set_yticklabels(HIRAC_CLASSES, fontsize=7, va="center")
                ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
                ax.set_xlabel("Time UTC")
            else:
                types = MAC_TYPES if label == "MAC" else HETEAC_TYPES
                type_var = "aerosol_classification" if label == "MAC" else "aerosol_type"
                plot_typing_curtain(source, types, label, type_var, ax=ax)
        except Exception as exc:  # noqa: BLE001 - one unreadable product must not lose the others
            ax.text(0.5, 0.5, f"{label}\ncould not be read\n({type(exc).__name__})",
                    ha="center", va="center", transform=ax.transAxes, fontsize=9)

        ax.set_title(label)
        if top_km:
            ax.set_ylim(0, top_km)

    # Each panel is built by a function that labels its own axes, so the wording differs from
    # one to the next; with a shared vertical axis only the leftmost label is needed.
    for ax in axes:
        ax.set_ylabel("")
    axes[0].set_ylabel("Altitude [km a.s.l.]")

    # Three panels side by side leave each one about a third of the width, and the time
    # formatter these products use writes the date underneath as well: without thinning the
    # ticks and tilting them the labels run into each other.
    for ax in axes:
        # These are date axes: a generic numeric locator would relabel them 12, 24, 36 ...
        locator = mdates.AutoDateLocator(minticks=2, maxticks=4)
        ax.xaxis.set_major_locator(locator)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(locator))
        for label in ax.get_xticklabels():
            label.set_rotation(30)
            label.set_horizontalalignment("right")
        # The date that ConciseDateFormatter writes in the corner repeats in every panel and
        # is already in the measurement identifier above.
        ax.xaxis.get_offset_text().set_visible(False)

    fig.suptitle(f"Aerosol typing - {STATION.upper()} - "
                 f"measurement {SELECTED_MEASUREMENT_ID}", fontsize=12)
    fig.tight_layout(rect=(0, 0, 1, 0.93))
    register_figure(fig, f"typing_comparison_{SELECTED_MEASUREMENT_ID}")
    plt.show()
    return fig


# The file lists come from the sections above; whichever is missing is simply left out.
# The result is assigned rather than left as the last expression: plot_typing_curtain and the
# cell below both display it otherwise, and the figure appears twice.
_hirac_candidates = globals().get("hirac_candidates") or []
_comparison = compare_typing(
    hirac_path=str(_hirac_candidates[0]) if _hirac_candidates else None,
    mac_paths=globals().get("mac_files"),
    heteac_paths=globals().get("heteac_files"),
)

## Save the remote sensing results

The lidar part of the work is finished here. The in-situ sections that follow are optional, still
in beta, and can take a while — the NRT one crawls two catalogs — so this is a good place to put
the optical profiles and the typing figures somewhere safe before going on.

Saving here does not end anything: the last section of the notebook offers the same controls
again, and writes a separate archive holding everything, in-situ included.

In [ ]:
# Everything this run downloaded or produced lives in `outdir`. JupyterLab can only download
# one file at a time, so the cell below packs the folder into a single zip next to the notebook
# and gives you a link to it.

import os
import zipfile

from IPython.display import HTML

FIGURE_SUFFIXES = (".png", ".pdf", ".svg", ".txt", ".csv")


def _folder_size(folder, suffixes=None):
    """Total size in bytes of the files that would go into the archive."""
    return sum(f.stat().st_size for f in Path(folder).rglob("*")
               if f.is_file() and (suffixes is None or f.suffix.lower() in suffixes))


def _human(n):
    for unit in ("B", "kB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.0f} {unit}" if unit == "B" else f"{n:.1f} {unit}"
        n /= 1024


def make_export_controls(tag, note=""):
    """A dropdown, a button and an output packing this run into <run>_<tag>.zip.

    Built by a function rather than written out once: the notebook saves twice, here with the
    remote sensing work finished and again at the end after the in-situ sections. Each call
    writes its own archive, so the second never overwrites the first.
    """
    bundle_dd = Dropdown(
    options=[("Figures and tables only (PNG, CSV, TXT)", "figures"),
             ("Everything, including the downloaded data", "all")],
    value="figures", description="Include:",
    style={"description_width": "initial"}, layout=Layout(width="480px"))
    bundle_btn = Button(description="Create archive", button_style="success", icon="file-archive-o")
    bundle_out = Output()


    def _make_bundle(_=None):
        with bundle_out:
            bundle_out.clear_output()
            if not outdir.exists():
                print(f"Nothing to pack: {outdir} does not exist yet.")
                return

            figures_only = bundle_dd.value == "figures"
            suffixes = FIGURE_SUFFIXES if figures_only else None

            # Write out any figure that was drawn but not saved, so the archive is complete even
            # if saving was left off while plotting.
            written_now = []
            for stem, fig in DRAWN_FIGURES.items():
                path = outdir / f"{stem}.png"
                if not path.exists():
                    fig.savefig(path, dpi=150, bbox_inches="tight")
                    written_now.append(path.name)
            if written_now:
                print(f"Wrote {len(written_now)} figure(s) that had not been saved yet: "
                      f"{', '.join(written_now)}")
            files = [f for f in sorted(outdir.rglob("*"))
                     if f.is_file() and (suffixes is None or f.suffix.lower() in suffixes)]
            if not files:
                print("Nothing to pack: no figure has been drawn yet in this session, and the run "
                      "folder holds no PNG, CSV or TXT. Run the plot cells above first.")
                return

            # The archive is written next to the notebook, so the download link below resolves
            archive = EXPORT_DIR / f"{outdir.name}_{tag}{'_figures' if figures_only else ''}.zip"
            with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as zf:
                for f in files:
                    zf.write(f, f.relative_to(outdir.parent))

            print(f"{len(files)} file(s), {_human(_folder_size(outdir, suffixes))} "
                  f"-> {_human(archive.stat().st_size)} compressed")
            for f in files[:10]:
                print(f"   {f.relative_to(outdir)}")
            if len(files) > 10:
                print(f"   ... and {len(files) - 10} more")
            # FileLink produces a link that JupyterLab tries to *open* in the editor, which fails
            # on a zip with "is not UTF-8 encoded". The /files/ endpoint with download=1 hands the
            # file to the browser instead. The path is relative to the Jupyter root, i.e. the home
            # directory.
            try:
                rel = archive.resolve().relative_to(Path.home())
            except ValueError:
                rel = archive.name
            # Under JupyterHub every URL is prefixed with /user/<name>/. Without that prefix the
            # link hits the hub instead of this server, which answers with an HTML error page -
            # the browser then saves that page as a .html file instead of the archive.
            _prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "/").rstrip("/")
            url = f"{_prefix}/files/{rel}?download=1"
            display(HTML(
                f'<p style="margin-top:8px">'
                f'<a href="{url}" download target="_blank" '
                f'style="padding:8px 14px;background:#1B4053;color:white;text-decoration:none;'
                f'border-radius:5px;display:inline-block">Download {archive.name}</a></p>'
                f'<p style="font-size:90%;color:#555">If the link does nothing, right-click '
                f'<code>{archive.name}</code> in the file browser on the left and choose '
                f'<i>Download</i>.</p>'))


    bundle_dd.observe(lambda _c: bundle_out.clear_output(), names="value")
    bundle_btn.on_click(_make_bundle)
    display(VBox([bundle_dd, bundle_btn, bundle_out]))
    if note:
        print(note)


make_export_controls(
    "remote_sensing",
    "Everything drawn so far goes in. You can save again after the in-situ sections: that "
    "writes a second archive and leaves this one alone.")

## Is there in-situ data at this station too? <span style="background-color: #d35400; color: white; padding: 2px 8px; border-radius: 4px; font-size: 0.6em; vertical-align: middle;">BETA</span>

<div style="border-left: 4px solid #d35400; background-color: #fbeee6; padding: 8px 12px; margin: 6px 0 12px;">
<b>Beta.</b> This section and the ones that follow it, up to the figure composer, are still being
tested and developed. They work on the cases they were tried on, such as Potenza on 5 June 2025, but
results may be incomplete and the interface may change. Please report anything that looks wrong.
</div>

A lidar sees the aerosol along the column; an in-situ instrument measures what reaches the ground.
At several ACTRIS sites both exist, and looking at the two together is often what turns a layer seen
aloft into a statement about air quality.

The check below asks the ACTRIS metadata archive which facilities sit near the station selected
above, and how far away they are. Proximity is not the same as belonging to the same site: two
facilities a few kilometres apart can differ in altitude by a thousand metres, which matters for
aerosol. The distance is printed so that you can judge.

Nothing is downloaded here. If a facility turns up, the in-situ notebooks of this VRE are the place
to go: `ACTRIS_aerosol_insitu_L2.ipynb` for quality-controlled data and
`ACTRIS_aerosol_insitu_NRT.ipynb` for near real time.

In [ ]:
def find_insitu_nearby(_=None):
    with insitu_out:
        insitu_out.clear_output()
        station = stations_by_id.get(STATION, {})
        lat, lon = station.get("Latitude"), station.get("Longitude")
        if lat is None or lon is None:
            print(f"The archive gives no coordinates for {STATION}, so nothing can be matched.")
            return

        print(f"{STATION} - {station.get('Name', '')} at {lat:.3f}, {lon:.3f}")
        print(f"Looking for ACTRIS facilities within {NEARBY_RADIUS_KM} km ...")
        try:
            response = requests.get(FACILITIES_URL, timeout=60)
            response.raise_for_status()
            facilities = response.json()
        except Exception as exc:  # noqa: BLE001 - a service that is down must not break the run
            print(f"The facility archive could not be reached ({type(exc).__name__}: {exc}).")
            print("This says nothing about the station - try again later.")
            return

        nearby = []
        for record in facilities:
            flat, flon = record.get("lat"), record.get("lon")
            if flat is None or flon is None:
                continue
            distance = _haversine_km(lat, lon, float(flat), float(flon))
            if distance <= NEARBY_RADIUS_KM:
                nearby.append((distance, record))
        nearby.sort(key=lambda item: item[0])
        global _insitu_nearby
        _insitu_nearby = nearby          # read by the selection menus further down

        if not nearby:
            print(f"\nNo ACTRIS facility within {NEARBY_RADIUS_KM} km.")
            print("Raise NEARBY_RADIUS_KM if you want to look further out; bear in mind that a")
            print("facility tens of kilometres away is a different site, not the same one.")
            return

        print(f"\n{len(nearby)} facility(ies) found:\n")
        for distance, record in nearby:
            name = _facility_field(record, "name", "facility_name", default="(unnamed)")
            code_ = _facility_field(record, "identifier", "code", "actris_id")
            altitude = _facility_field(record, "alt", "altitude", default=None)
            national = str(_facility_field(record, "actris_national_facility",
                                           default="")).lower() == "true"
            line = f"  {distance:5.1f} km   {name}"
            if code_:
                line += f"  [{code_}]"
            print(line)
            details = []
            if altitude not in (None, ""):
                details.append(f"altitude {float(altitude):.0f} m")
                if station.get("Altitude") is not None:
                    details.append(f"lidar station at {float(station['Altitude']):.0f} m")
            details.append("National Facility" if national else "not labelled as National Facility")
            print(f"            {', '.join(details)}")

        print("\nTo see what these facilities actually measured, open the in-situ notebooks:")
        print("  ../aerosol_in_situ/in_situ_L2/ACTRIS_aerosol_insitu_L2.ipynb    (quality-controlled)")
        print("  ../aerosol_in_situ/in_situ_NRT/ACTRIS_aerosol_insitu_NRT.ipynb  (near real time)")
        print("A facility appearing here does not mean it has aerosol in-situ data for your period:")
        print("that is the question those notebooks answer.")


insitu_btn = Button(description="Check for in-situ facilities", button_style="info",
                    icon="search", layout=Layout(width="260px"))
insitu_out = Output()
insitu_btn.on_click(find_insitu_nearby)
display(VBox([insitu_btn, insitu_out]))

with insitu_out:
    print("Press the button to query the ACTRIS facility archive.")

### Load the in-situ module

The in-situ notebooks of this VRE do their work through `ACTRIS_insitu.py`, which lives in
`actris/aerosol_in_situ/in_situ_L2/`. Rather than repeating its queries here, that module is
imported and driven directly, so the selection offered below is the one its own notebook offers,
and a fix made there arrives here too.

The folder is found by walking up from this notebook, so it works from any copy of the VRE as long
as the folders are kept together. If it is not found, the cell says so and the rest of the notebook
is unaffected.

In [ ]:
INSITU_MODULE_DIR = None


def _find_insitu_dir(start=None):
    """The folder holding ACTRIS_insitu.py, looked for next to this notebook's VRE copy."""
    here = Path(start or os.getcwd()).resolve()
    for base in [here, *here.parents][:6]:
        candidate = base / "aerosol_in_situ" / "in_situ_L2" / "ACTRIS_insitu.py"
        if candidate.is_file():
            return candidate.parent
        candidate = base / "actris" / "aerosol_in_situ" / "in_situ_L2" / "ACTRIS_insitu.py"
        if candidate.is_file():
            return candidate.parent
    return None


# ACTRIS_insitu.py imports ebas_plot and tqdm at module level. They are dependencies of the
# in-situ notebooks, not of this one, so they are installed here only when this optional
# section is actually used - a full run that never comes down here pays nothing for them.
INSITU_REQUIRES = {"ebas_plot": "ebas-plot", "tqdm": "tqdm"}


def _ensure_insitu_requirements():
    missing = [pkg for mod, pkg in INSITU_REQUIRES.items()
               if importlib.util.find_spec(mod) is None]
    if not missing:
        return True
    print(f"The in-situ module needs {', '.join(missing)} - installing ...")
    result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                            capture_output=True, text=True, cwd=os.path.expanduser("~"))
    if result.returncode != 0:
        print("Installation failed:")
        print(result.stderr[-800:])
        return False
    # A package installed after the interpreter started is not on the import machinery's
    # radar until the caches are refreshed.
    importlib.invalidate_caches()
    print("Done.")
    return True


insitu = None
INSITU_MODULE_DIR = _find_insitu_dir()
if INSITU_MODULE_DIR is None:
    print("ACTRIS_insitu.py not found next to this notebook.")
    print("The in-situ section below will not work; the rest of the notebook is unaffected.")
    print("It is expected in actris/aerosol_in_situ/in_situ_L2/ of this VRE.")
else:
    if str(INSITU_MODULE_DIR) not in sys.path:
        sys.path.insert(0, str(INSITU_MODULE_DIR))
    try:
        if not _ensure_insitu_requirements():
            raise ImportError("the packages the in-situ module needs are not available")
        import ACTRIS_insitu as insitu
        print(f"In-situ module loaded from {INSITU_MODULE_DIR}")
        print("Reading the facility and dataset metadata (a few seconds) ...")
        insitu.load_metadata()
    except Exception as exc:  # noqa: BLE001 - its dependencies are not this notebook's business
        insitu = None
        print(f"The in-situ module could not be loaded ({type(exc).__name__}: {exc}).")
        print("Open ACTRIS_aerosol_insitu_L2.ipynb once: it installs what it needs.")

### Choose what to look at, and plot it over the lidar window

The menus are the same as in the in-situ notebook: instrument, variable, facility, instrument
model. The facility list is **not** filtered to the ones found above — a station may have in-situ
data under a name that does not match, and hiding the rest would make that invisible — but the
facilities within reach of the lidar are marked.

The plot covers the period selected at the top of this notebook, with the measurement itself
shaded, so that what the lidar saw aloft and what the instrument measured at the ground sit on the
same time axis.

In [ ]:
if insitu is None:
    print("The in-situ module is not available - see the cell above.")
else:

    _nearby_names = {str(_facility_field(r, "name", "facility_name")).lower()
                     for _d, r in (globals().get("_insitu_nearby") or [])}

    instrument_dd = Dropdown(options=list(insitu._TARGETS), description="Instrument:",
                             style={"description_width": "initial"},
                             layout=Layout(width="460px"))
    variable_dd = Dropdown(options=[], description="Variable:",
                           style={"description_width": "initial"},
                           layout=Layout(width="460px"))
    facility_dd = Dropdown(options=[], description="Facility:",
                           style={"description_width": "initial"},
                           layout=Layout(width="460px"))
    model_dd = Dropdown(options=[], description="Model:",
                        style={"description_width": "initial"},
                        layout=Layout(width="460px"))
    only_nearby_cb = Checkbox(
        value=bool(_nearby_names),
        description=f"Only facilities near the lidar station ({len(_nearby_names)} found)",
        indent=False, layout=Layout(width="460px"))
    with_data_cb = Checkbox(value=True, indent=False, layout=Layout(width="460px"),
                            description="Only facilities with data in the lidar period")
    scan_btn = Button(description="Check which have data", button_style="info", icon="search",
                      layout=Layout(width="240px"))
    scan_out = Output()
    insitu_plot_btn = Button(description="Load and plot", button_style="success",
                             icon="line-chart")
    insitu_plot_out = Output()


    # Facilities known to have data in the lidar period, filled by the scan below. Empty
    # means "not checked yet", which is not the same as "none" and must not be shown as such.
    _with_data = {}


    def _overlaps(document, start, end):
        """True unless the dataset's declared extent is entirely outside the window."""
        extent = document.get("temporal_extent") or {}
        begin, finish = extent.get("time_period_begin"), extent.get("time_period_end")
        try:
            d0 = np.datetime64(begin[:19]) if begin else None
            d1 = np.datetime64(finish[:19]) if finish else None
        except (TypeError, ValueError):
            return True          # undated: it cannot be ruled out
        return (d0 is None or d0 <= end) and (d1 is None or d1 >= start)


    def _documents(instrument, variable, facility, model=None, start=None, end=None):
        """Documents matching this selection, optionally restricted to a model and a period.

        The marking in the menus and the loading below both go through this, because when they
        each had their own query they disagreed: one said eight datasets, the other none.
        """
        documents = insitu.search(
            insitu.PROD_API,
            insitu._dataset_query(instrument, variable, facility, size=1000))
        if model:
            documents = [d for d in documents if insitu._matches_model(d, model)]
        if start is not None:
            documents = [d for d in documents if _overlaps(d, start, end)]
        return documents


    def scan_period(_=None):
        """Which facilities have data for this instrument/variable in the lidar period.

        One query covers every facility at once: asking per facility would mean dozens of
        requests to answer a question the metadata already carries.
        """
        with scan_out:
            scan_out.clear_output()
            instrument, variable = instrument_dd.value, variable_dd.value
            if not (instrument and variable):
                return
            start = np.datetime64(f"{FROM_DATE}T{FROM_TIME}") - np.timedelta64(2, "D")
            end = np.datetime64(f"{TO_DATE}T{TO_TIME}") + np.timedelta64(2, "D")
            print(f"Checking which facilities measured {variable} between "
                  f"{str(start)[:10]} and {str(end)[:10]} ...")
            try:
                documents = insitu.search(
                    insitu.PROD_API, insitu._dataset_query(instrument, variable, size=1000))
            except Exception as exc:  # noqa: BLE001
                print(f"The metadata search failed ({type(exc).__name__}: {exc}).")
                return

            counts = {}
            for document in documents:
                name = ((document.get("facility") or {}).get("name"))
                if name and _overlaps(document, start, end):
                    counts[name] = counts.get(name, 0) + 1

            _with_data.clear()
            _with_data.update(counts)
            if not counts:
                print("No facility has data for this variable in that period.")
            else:
                near = [n for n in counts if n.lower() in _nearby_names]
                print(f"{len(counts)} facility(ies) have data, "
                      f"{len(near)} of them near the lidar station.")
                print("The counts cover every instrument model at that facility; the Model "
                      "menu below says which of them actually measured in your period.")
            _refresh_facilities()


    def _label_facility(name):
        """Mark the facilities that sit near the lidar station."""
        marks = []
        if name.lower() in _nearby_names:
            marks.append("near the lidar station")
        if _with_data:
            marks.append(f"{_with_data[name]} dataset(s) in your period, any model"
                         if name in _with_data else "no data in your period")
        return f"{name}  ({', '.join(marks)})" if marks else name


    def _refresh_variables(_=None):
        variables = list(insitu._TARGETS.get(instrument_dd.value, {}))
        variable_dd.options = variables
        if variables:
            variable_dd.value = variables[0]


    def _refresh_facilities(_=None):
        if not (instrument_dd.value and variable_dd.value):
            return
        names = insitu._stations_with_coords(instrument_dd.value, variable_dd.value)
        near = [n for n in names if n.lower() in _nearby_names]
        rest = [n for n in names if n.lower() not in _nearby_names]
        # With the box ticked only the facilities found by the proximity check are offered.
        # The full list stays one click away: a site can publish in-situ data under a name
        # the check does not match, and hiding that for good would be worse than a long list.
        shown = near if (only_nearby_cb.value and near) else near + rest
        if with_data_cb.value and _with_data:
            kept = [n for n in shown if n in _with_data]
            shown = kept or shown      # never leave the menu empty
        facility_dd.options = [(_label_facility(n), n) for n in shown]
        if facility_dd.options:
            facility_dd.value = facility_dd.options[0][1]


    def _refresh_models(_=None):
        """List the models, marking those that measured during the lidar period.

        Without this the facility can promise "2 datasets in your period" and the plot then
        find none: the promise covers the facility, the plot covers one model of it.
        """
        if not (instrument_dd.value and variable_dd.value and facility_dd.value):
            return
        try:
            models = insitu._fetch_models(instrument_dd.value, variable_dd.value,
                                          facility_dd.value)
        except Exception as exc:  # noqa: BLE001
            with insitu_plot_out:
                print(f"Could not list the instrument models ({type(exc).__name__}).")
            model_dd.options = []
            return

        start = np.datetime64(f"{FROM_DATE}T{FROM_TIME}") - np.timedelta64(2, "D")
        end = np.datetime64(f"{TO_DATE}T{TO_TIME}") + np.timedelta64(2, "D")
        per_model = {}
        try:
            for document in _documents(instrument_dd.value, variable_dd.value,
                                       facility_dd.value, start=start, end=end):
                # One document can carry the same instrument once per variable - seven
                # wavelengths of an absorption product, say - so the models it mentions are
                # collected first and counted once each. Counting the entries themselves is
                # what turned one dataset into "8 dataset(s)".
                keys = {insitu._model_key(label)
                        for variable_data in document.get("variables", [])
                        for instrument_data in variable_data.get("instrument", [])
                        if (label := insitu._model_label(instrument_data))}
                for key in keys:
                    per_model[key] = per_model.get(key, 0) + 1
        except Exception:  # noqa: BLE001 - the plain list is still usable without the marks
            per_model = {}

        options = []
        for entry in models:
            label, value = entry if isinstance(entry, tuple) else (entry, entry)
            count = per_model.get(insitu._model_key(value))
            suffix = (f"  -  {count} dataset(s) in your period" if count
                      else "  -  nothing in your period" if per_model else "")
            options.append((f"{label}{suffix}", value))
        model_dd.options = options
        if options:
            # Open on a model that has data, when there is one: picking the first
            # alphabetically is what made the plot come back empty.
            with_data = [v for _l, v in options if per_model.get(insitu._model_key(v))]
            model_dd.value = with_data[0] if with_data else options[0][1]


    def _plot_insitu(_=None):
        with insitu_plot_out:
            insitu_plot_out.clear_output()
            state = {"instrument": instrument_dd.value, "variable": variable_dd.value,
                     "station": facility_dd.value, "model": model_dd.value, "doi": None}
            start = np.datetime64(f"{FROM_DATE}T{FROM_TIME}")
            end = np.datetime64(f"{TO_DATE}T{TO_TIME}")
            margin = np.timedelta64(2, "D")
            print(f"{state['station']}: {state['variable']} ({state['instrument']})")

            # A facility's record can span more than a decade. Loading all of it and then
            # cutting out two days is what kills the kernel, so the datasets are filtered on
            # the temporal extent the metadata already carries, and only the ones overlapping
            # the lidar period are opened.
            try:
                all_docs = _documents(state["instrument"], state["variable"],
                                      state["station"], model=state["model"])
                in_period = _documents(state["instrument"], state["variable"],
                                       state["station"], model=state["model"],
                                       start=start - margin, end=end + margin)
            except Exception as exc:  # noqa: BLE001
                print(f"The metadata search failed ({type(exc).__name__}: {exc}).")
                return

            # The identifier lives at identification.identifier.pid - the same path the
            # module's own find_datasets() uses. Reading it from anywhere else returns
            # nothing, which looked like "no data" when the data were there all along.
            wanted = []
            for document in in_period:
                pid = ((document.get("identification") or {})
                       .get("identifier") or {}).get("pid")
                if pid:
                    wanted.append(pid)

            print(f"{len(all_docs)} dataset(s) for this model, "
                  f"{len(in_period)} covering the lidar period.")
            if not in_period:
                print("None of them covers it: this model has no data for those days.")
                print("The Model menu above marks the ones that do.")
                return
            if not wanted:
                print("They carry no identifier this notebook can resolve, so nothing can be "
                      "opened - worth reporting to the data centre.")
                return
            if len(wanted) > MAX_INSITU_DATASETS:
                print(f"Opening the {MAX_INSITU_DATASETS} most relevant of them "
                      f"(MAX_INSITU_DATASETS limits this).")
                wanted = wanted[:MAX_INSITU_DATASETS]

            parts = []
            for doi in wanted:
                try:
                    part = xr.open_dataset(insitu._doi_to_opendap_url(doi))
                    parts.append(insitu._clean_dataset(part))
                except Exception as exc:  # noqa: BLE001 - one bad file must not lose the rest
                    print(f"   could not open {doi}: {type(exc).__name__}")
            if not parts:
                print("None of the datasets could be opened.")
                return
            ds = parts[0] if len(parts) == 1 else xr.concat(parts, dim="time")

            window = ds.sel(time=slice(start - margin, end + margin))
            if window.time.size == 0:
                print(f"The record exists but has no measurement between "
                      f"{start} and {end}.")
                print(f"It covers {str(ds.time.min().values)[:16]} to "
                      f"{str(ds.time.max().values)[:16]}.")
                return

            time_name = _time_coordinate(window)
            plotted = _time_series(window, time_name) if time_name else []
            if not plotted:
                print("Nothing plottable in this period. What the file holds:")
                for name, variable in list(window.data_vars.items())[:12]:
                    finite = int(np.isfinite(variable).sum()) if variable.size else 0
                    print(f"   {name}  dims={variable.dims}  {finite} finite value(s)")
                if time_name is None:
                    print("No time coordinate was recognised, which is the likely reason.")
                return

            fig, ax = plt.subplots(figsize=(12, 4.5))
            for label, series_data in plotted:
                ax.plot(window[time_name], series_data, linewidth=0.9, label=label)
            ax.axvspan(start, end, color="tab:orange", alpha=0.15,
                       label="lidar measurement period")
            ax.set_xlabel("Time (UTC)")
            ax.set_ylabel(plotted[0][1].attrs.get("units", ""))
            ax.set_title(f"{state['station']} - {state['variable']}\n"
                         f"in-situ, around the lidar measurement at {STATION.upper()}")
            ax.legend(fontsize=8)
            ax.grid(alpha=0.3)
            fig.tight_layout()
            register_figure(fig, f"insitu_{state['station']}_{state['variable']}")
            plt.show()


    scan_btn.on_click(scan_period)
    only_nearby_cb.observe(_refresh_facilities, names="value")
    with_data_cb.observe(_refresh_facilities, names="value")
    instrument_dd.observe(_refresh_variables, names="value")
    variable_dd.observe(_refresh_facilities, names="value")
    facility_dd.observe(_refresh_models, names="value")
    insitu_plot_btn.on_click(_plot_insitu)

    _refresh_variables()
    _refresh_facilities()
    _refresh_models()
    display(VBox([instrument_dd, variable_dd,
                  only_nearby_cb, with_data_cb, scan_btn, scan_out,
                  facility_dd, model_dd, insitu_plot_btn, insitu_plot_out]))
    print("Press 'Check which have data' to mark the facilities that measured this variable "
          "during your lidar period.")

### Near real time in-situ data <span style="background-color: #d35400; color: white; padding: 2px 8px; border-radius: 4px; font-size: 0.6em; vertical-align: middle;">BETA</span>

The section above uses the quality-controlled (Level 2) record, which lags months behind the
measurement. For a recent case the near real time products are the ones that exist, and they are
reached differently: not through the metadata API but by crawling two THREDDS catalogs, which takes
a minute or two and is why it sits behind a button rather than running on its own.

This reuses the modules of `ACTRIS_aerosol_insitu_NRT.ipynb`, so the interface below is the one
that notebook offers — instrument, variable, facility, file, and a time window. Pick a file, press
its **Plot** button, and the last cell of this section can then redraw the same series restricted to
the period of your lidar measurement.

In [ ]:
NRT_REQUIRES = {"threddsclient": "threddsclient", "plotly": "plotly", "narwhals": "narwhals"}
THREDDS_ACTRIS_NRT = "https://thredds.nilu.no/thredds/catalog/actris_nrt/catalog.xml"
THREDDS_EBAS = "https://thredds.nilu.no/thredds/catalog/ebas/catalog.xml"

nrt_ui_module = None
nrt_app_out = Output()


def _find_nrt_src(start=None):
    """The nrt_src folder of the in-situ NRT notebook."""
    here = Path(start or os.getcwd()).resolve()
    for base in [here, *here.parents][:6]:
        for candidate in (base / "aerosol_in_situ" / "in_situ_NRT" / "nrt_src",
                          base / "actris" / "aerosol_in_situ" / "in_situ_NRT" / "nrt_src"):
            if (candidate / "nrt_ui.py").is_file():
                return candidate
    return None


def load_nrt(_=None):
    global nrt_ui_module
    with nrt_app_out:
        nrt_app_out.clear_output()
        src_dir = _find_nrt_src()
        if src_dir is None:
            print("nrt_src not found: it is expected in actris/aerosol_in_situ/in_situ_NRT/.")
            return

        missing = [pkg for mod, pkg in NRT_REQUIRES.items()
                   if importlib.util.find_spec(mod) is None]
        if missing:
            print(f"Installing what the NRT modules need: {', '.join(missing)} ...")
            result = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *missing],
                                    capture_output=True, text=True, cwd=os.path.expanduser("~"))
            if result.returncode != 0:
                print("Installation failed:")
                print(result.stderr[-800:])
                return
            importlib.invalidate_caches()

        # Two folders, not one: nrt_config, nrt_utils and nrt_ui live in nrt_src/, while
        # actris_plotting_utilities.py sits one level up, next to the NRT notebook itself,
        # and nrt_ui imports it by plain name.
        for folder in (src_dir, src_dir.parent):
            if str(folder) not in sys.path:
                sys.path.insert(0, str(folder))
        try:
            import threddsclient
            import nrt_utils as nu
            import nrt_ui as nui
        except Exception as exc:  # noqa: BLE001
            print(f"The NRT modules could not be imported ({type(exc).__name__}: {exc}).")
            return

        print("Reading the two THREDDS catalogs - this is the slow part, a minute or two ...")
        try:
            actris_urls = threddsclient.opendap_urls(THREDDS_ACTRIS_NRT)
            ebas_urls = [u for u in threddsclient.opendap_urls(THREDDS_EBAS)
                         if nu.is_nrt_file(u)]
        except Exception as exc:  # noqa: BLE001 - the server is often the reason
            print(f"A catalog could not be read ({type(exc).__name__}: {exc}).")
            print("Nothing else in this notebook depends on it.")
            return
        print(f"{len(actris_urls)} actris_nrt and {len(ebas_urls)} ebas dataset(s) found.")

        df_files, *_rest = nu.build_combined_file_index(actris_urls, ebas_urls,
                                                        show_tables=False)
        if len(df_files) == 0:
            print("The combined index is empty - there is nothing to explore.")
            return
        print(f"Index built: {len(df_files)} unique NRT files.\n")

        # Same two questions as the Level 2 section above, answered before the user starts
        # clicking: which facilities are near the lidar station, and which of them have a
        # file covering its period. The index already carries both, so this costs nothing.
        start = np.datetime64(f"{FROM_DATE}T{FROM_TIME}") - np.timedelta64(2, "D")
        end = np.datetime64(f"{TO_DATE}T{TO_TIME}") + np.timedelta64(2, "D")
        station = stations_by_id.get(STATION, {})
        lat, lon = station.get("Latitude"), station.get("Longitude")

        # Counted as distinct files, not as index rows: one file reporting several variables
        # appears once per variable in the index, and counting rows inflated the total.
        in_period_files, near_codes = {}, {}
        best_selection = None        # instrument/variable/facility to open the menus on
        for _i, row in df_files.iterrows():
            code = row.get("station_code") or row.get("station") or row.get("facility")
            if not code:
                continue
            begin = row.get("start_dt")
            finish = row.get("end_dt", begin)
            try:
                d0 = np.datetime64(begin) if begin is not None else None
                d1 = np.datetime64(finish) if finish is not None else d0
            except (TypeError, ValueError):
                d0 = d1 = None
            if (d0 is None or d0 <= end) and (d1 is None or d1 >= start):
                in_period_files.setdefault(code, set()).add(row.get("opendap_url", _i))

        if lat is not None and lon is not None:
            for code, (flat, flon) in getattr(nui, "_STATION_COORDS", {}).items():
                distance = _haversine_km(lat, lon, flat, flon)
                if distance <= NEARBY_RADIUS_KM:
                    near_codes[code] = distance

        print(f"Your lidar measurement: {FROM_DATE} {FROM_TIME} -> {TO_DATE} {TO_TIME}")
        in_period = {code: len(files) for code, files in in_period_files.items()}
        if in_period:
            print(f"{len(in_period)} facility(ies) have NRT files covering it.")
        else:
            print("No NRT file covers that period - the catalogs hold the recent weeks only,")
            print("so an older case will not be found here.")
        if near_codes:
            print(f"\nWithin {NEARBY_RADIUS_KM} km of {STATION}:")
            for code, distance in sorted(near_codes.items(), key=lambda kv: kv[1]):
                name = getattr(nui, "_STATION_NAMES", {}).get(code, "")
                files = in_period.get(code, 0)
                state = f"{files} file(s) in your period" if files else "nothing in your period"
                print(f"   {distance:5.1f} km  {code} {name} - {state}")

                # A file count alone does not say what was measured, and the menus below
                # offer every instrument whether or not it has anything in the period. The
                # breakdown is built with the module's own selectors, so it matches exactly
                # what those menus will show.
                if not files:
                    continue
                try:
                    for instrument in nu.get_nrt_instruments(df_files):
                        # Grouped by instrument, with the variables listed and the files
                        # counted once. A nephelometer reports scattering and backscattering
                        # from the same file, so counting per variable showed "1 file" twice
                        # and read as two.
                        variables, seen_files = [], set()
                        for variable in nu.get_nrt_attributes_for_instrument(instrument):
                            if code not in nu.get_nrt_stations_for_selection(
                                    df_files, instrument, variable):
                                continue
                            selection = nu.filter_nrt_files(df_files, instrument, variable, code)
                            kept = 0
                            for key, row in selection.iterrows():
                                begin = row.get("start_dt")
                                finish = row.get("end_dt", begin)
                                try:
                                    e0 = np.datetime64(begin) if begin is not None else None
                                    e1 = np.datetime64(finish) if finish is not None else e0
                                except (TypeError, ValueError):
                                    e0 = e1 = None
                                if (e0 is None or e0 <= end) and (e1 is None or e1 >= start):
                                    kept += 1
                                    seen_files.add(row.get("opendap_url", key))
                            if kept:
                                variables.append(variable)
                                if best_selection is None:
                                    best_selection = (instrument, variable, code)
                        if variables:
                            print(f"        {instrument}: {len(seen_files)} file(s) - "
                                  f"{', '.join(variables)}")
                except Exception as exc:  # noqa: BLE001 - the summary is a convenience
                    print(f"        (could not break it down: {type(exc).__name__})")
        elif lat is not None:
            print(f"\nNo NRT facility within {NEARBY_RADIUS_KM} km of {STATION}.")
        print("\nChoose a file in the interface below, then press its Plot button.\n")

        nrt_ui_module = nui
        # Open the menus on a combination that actually has files in the period, rather than
        # on whatever comes first: that is what made the interface look empty.
        if best_selection:
            instrument, variable, code = best_selection
            defaults = {"instrument": instrument, "variable": variable, "facility": code}
            print(f"Opening on {code} / {instrument} / {variable}.")
        else:
            _best = sorted((d, c) for c, d in near_codes.items() if c in in_period)
            defaults = {"facility": _best[0][1]} if _best else None
        display(nui.build_app(df_files, defaults=defaults) if defaults
                else nui.build_app(df_files))


nrt_btn = Button(description="Load the NRT index", button_style="info", icon="download",
                 layout=Layout(width="220px"))
nrt_btn.on_click(load_nrt)
display(VBox([nrt_btn, nrt_app_out]))

Once a file is plotted above, the cell below draws the same series again over the period of the
lidar measurement, with the measurement itself shaded. NRT files cover a day or a few days, so if
the window falls outside the file the cell says so rather than drawing an empty axis.

In [ ]:
def plot_nrt_over_window(_=None):
    with nrt_window_out:
        nrt_window_out.clear_output()
        if nrt_ui_module is None:
            print("Load the NRT index first, with the button above.")
            return
        ds = nrt_ui_module.current_dataset()
        if ds is None:
            print("No NRT dataset loaded yet: choose a file in the interface above and press Plot.")
            return

        start = np.datetime64(f"{FROM_DATE}T{FROM_TIME}")
        end = np.datetime64(f"{TO_DATE}T{TO_TIME}")
        margin = np.timedelta64(12, "h")
        window = ds.sel(time=slice(start - margin, end + margin))
        if window.time.size == 0:
            print(f"This file does not cover {start} to {end}.")
            print(f"It runs from {str(ds.time.min().values)[:16]} to "
                  f"{str(ds.time.max().values)[:16]} - pick another one above.")
            return

        time_name = _time_coordinate(window)
        plotted = _time_series(window, time_name) if time_name else []
        if not plotted:
            print("Nothing plottable in this period. What the file holds:")
            for name, variable in list(window.data_vars.items())[:12]:
                finite = int(np.isfinite(variable).sum()) if variable.size else 0
                print(f"   {name}  dims={variable.dims}  {finite} finite value(s)")
            return

        fig, ax = plt.subplots(figsize=(12, 4.5))
        for label, series_data in plotted:
            ax.plot(window[time_name], series_data, linewidth=0.9, label=label)
        ax.axvspan(start, end, color="tab:orange", alpha=0.15, label="lidar measurement period")
        ax.set_xlabel("Time (UTC)")
        ax.set_ylabel(plotted[0][1].attrs.get("units", ""))
        ax.set_title(f"NRT in-situ around the lidar measurement at {STATION.upper()}")
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        fig.tight_layout()
        register_figure(fig, f"insitu_nrt_{STATION}")
        plt.show()


nrt_window_btn = Button(description="Plot over the lidar window", button_style="success",
                        icon="line-chart", layout=Layout(width="240px"))
nrt_window_out = Output()
nrt_window_btn.on_click(plot_nrt_over_window)
display(VBox([nrt_window_btn, nrt_window_out]))

## Put a few figures side by side

Everything drawn above — the optical profiles, the typing panels, the in-situ series — is kept in
memory as it is produced. Pick two or three here and they are redrawn into a single figure, which
is easier to read than scrolling between cells and can be exported like any other.

The panels are images of the originals, not redrawn plots: axes and scales stay exactly as they
were, which is the point when the comparison is between a profile in kilometres and a time series
in hours.

In [ ]:
import io

figure_pick = SelectMultiple(
    options=list(DRAWN_FIGURES),
    rows=min(10, max(3, len(DRAWN_FIGURES))),
    description="Figures:", style={"description_width": "initial"},
    layout=Layout(width="760px"))
arrange_dd = Dropdown(
    options=[("One above the other", "rows"), ("Side by side", "columns")],
    value="rows", description="Arrange:", style={"description_width": "initial"},
    layout=Layout(width="360px"))
compose_btn = Button(description="Combine", button_style="success", icon="th-large")
compose_out = Output()


def _figure_as_image(fig, dpi=150):
    """A rendered copy of a figure, so it can be placed inside another one untouched."""
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", dpi=dpi, bbox_inches="tight")
    buffer.seek(0)
    return plt.imread(buffer)


def combine_figures(_=None):
    with compose_out:
        compose_out.clear_output()
        chosen = list(figure_pick.value)
        if len(chosen) < 2:
            print("Pick at least two figures - ctrl-click to add one.")
            return
        if len(chosen) > 4:
            print("Four panels is as much as stays readable; taking the first four.")
            chosen = chosen[:4]

        images = [(name, _figure_as_image(DRAWN_FIGURES[name])) for name in chosen]
        # Each panel keeps the aspect ratio of its original, so a tall profile next to a wide
        # time series does not end up stretched.
        # Panels of equal size, with each image keeping its own proportions inside its slot.
        # Sizing the slots by the images instead gave a tall lidar profile three quarters of
        # the page and squashed the time series next to it into a strip.
        ratios = [img.shape[0] / img.shape[1] for _n, img in images]
        tallest = max(ratios)
        if arrange_dd.value == "rows":
            width = 11.0
            height = min(16.0, width * tallest * len(images))
            fig, axes = plt.subplots(len(images), 1, figsize=(width, height))
        else:
            height = 6.0
            width = min(16.0, height / max(tallest, 0.1) * len(images))
            fig, axes = plt.subplots(1, len(images), figsize=(width, height))
        axes = np.atleast_1d(axes)

        for ax, (name, image) in zip(axes, images):
            ax.imshow(image)
            ax.set_title(name, fontsize=9)
            ax.axis("off")

        fig.tight_layout()
        stem = "comparison_" + "_".join(n[:18] for n in chosen)
        register_figure(fig, stem)
        plt.show()
        print(f"Combined {len(chosen)} figure(s). It is now in the export as {stem}.png")


def refresh_figure_list(_=None):
    """Pick up figures drawn after this cell was first run."""
    figure_pick.options = list(DRAWN_FIGURES)
    with compose_out:
        compose_out.clear_output()
        print(f"{len(DRAWN_FIGURES)} figure(s) available.")


refresh_btn = Button(description="Refresh list", icon="refresh", layout=Layout(width="150px"))
refresh_btn.on_click(refresh_figure_list)
compose_btn.on_click(combine_figures)
display(VBox([figure_pick, arrange_dd, HBox([compose_btn, refresh_btn]), compose_out]))

if not DRAWN_FIGURES:
    print("No figure has been drawn yet - run the plot cells above, then press 'Refresh list'.")
else:
    print(f"{len(DRAWN_FIGURES)} figure(s) available. Drawn something since? Press 'Refresh list'.")

<a id="export"></a>
## Take everything with you

Everything this run downloaded or produced is in the folder shown at the top of the notebook
(`data/<station>_<from>_<to>/`), which is **wiped at the start of the next run**. If you want to keep
any of it, download it before running the notebook again.

The cell below packs that folder into a single zip and gives you a download link. Two choices:

* **Figures and tables only** — the PNG, CSV and TXT files. Small, and usually what you want.
* **Everything** — the downloaded NetCDF products and archives as well. Useful to keep the raw data
  of a case study, but considerably larger.

The zip is written next to the notebook, so it also stays available in the file browser.

The archive is written into an `exports/` folder next to the notebook. That folder is emptied at the start of each run as well, so download what you need before running the notebook again.

This is the second saving point. It writes its own archive, named `..._complete`, so the one you may have taken after the remote sensing part is left untouched — keep both, or just this one if you went all the way through.

In [ ]:
# The same controls as the saving point after the typing comparison, writing a separate
# archive so that neither overwrites the other. Taken here, it holds everything the run
# produced, the in-situ figures included.
make_export_controls(
    "complete",
    "This archive holds everything produced in this run, the in-situ figures included.")